<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/NLP-2026/Lecture_5/%D0%9B%D0%B5%D0%BA%D1%86%D0%B8%D1%8F_5_1_%D0%92%D0%B2%D0%B5%D0%B4%D0%B5%D0%BD%D0%B8%D0%B5_%D0%B2_Retrieval_Augmented_Generation_(RAG).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лекция 5.1. Введение в Retrieval-Augmented Generation (RAG)



## Тема 1. Что такое RAG и зачем он нужен

### 1.1. Определение Retrieval-Augmented Generation

**Retrieval-Augmented Generation (RAG)** — это гибридная архитектура обработки естественного языка, которая объединяет два ключевых компонента: *информационный поиск (retrieval)* и *генерацию текста (generation)*. В отличие от классических поисковых систем, которые возвращают список документов или ссылок, RAG использует найденные документы как динамический контекст для генерации связного, фактологически обоснованного ответа на пользовательский запрос.

**Исторический контекст.** Концепция RAG была впервые предложена в 2020 году исследователями из Meta AI (тогда Facebook AI) в статье *«Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks»* (Lewis et al., 2020). Авторы показали, что комбинирование параметрической памяти (веса предобученной языковой модели) и непараметрической памяти (внешний индекс документов) позволяет значительно улучшить качество ответов на задачи, требующие глубоких предметных знаний. С тех пор RAG стал одним из наиболее востребованных подходов в области прикладного NLP, особенно после появления мощных открытых LLM и векторных баз данных.

**Три столпа RAG.** Архитектура RAG опирается на три фундаментальных компонента:

1. **Ретривер (Retriever)** — отвечает за поиск релевантных документов или фрагментов текста из внешнего корпуса по запросу пользователя. Обычно ретривер представляет собой модель эмбеддингов (например, *sentence‑transformers* или *BGE*), которая преобразует запрос и документы в векторное пространство, а затем выполняет поиск ближайших соседей с использованием косинусной близости или других метрик.

2. **Генератор (Generator)** — языковая модель (LLM), которая принимает на вход запрос и найденные ретривером документы (контекст) и генерирует итоговый ответ. Генератор может быть как локальной моделью (например, LLaMA, Qwen, Mistral), так и облачной (GPT‑4, Claude). Важно, чтобы модель имела достаточный размер контекстного окна для размещения всех релевантных фрагментов.

3. **Интеграция (Integration)** — механизм, связывающий ретривер и генератор. Он включает в себя формирование промпта, в который вставляются найденные документы (обычно с указанием источника), а также постобработку ответа (добавление ссылок, проверка фактов). Интеграция также может включать в себя дополнительные этапы, такие как переранжирование результатов, фильтрацию по метаданным, или даже многократные циклы поиска-генерации (как в Self‑RAG или Agentic RAG).

**Сравнение RAG с альтернативами.** Чтобы лучше понять место RAG в ландшафте технологий, сравним его с классическим поиском и с чистой LLM.

| Критерий | Классический поиск (например, Google) | Чистая LLM (без внешних данных) | RAG |
| :--- | :--- | :--- | :--- |
| **Источник знаний** | Индекс веб-страниц или внутренних документов | Исключительно веса модели (параметрическая память) | Внешние документы + параметрическая память |
| **Актуальность знаний** | Мгновенное обновление индекса (для веба – почти real‑time) | Фиксирована на момент обучения (срез данных) | Зависит от частоты обновления базы документов |
| **Формат ответа** | Список ссылок или кратких сниппетов | Связный текст, сгенерированный моделью | Связный текст с возможными ссылками на источники |
| **Галлюцинации** | Отсутствуют (поиск не генерирует факты) | Высокий риск галлюцинаций (особенно на узкие темы) | Значительно ниже, чем у чистой LLM, за счёт привязки к контексту |
| **Прозрачность** | Всегда видно источники | Невозможно проверить, откуда взята информация | Можно явно указать, из каких документов взят факт |
| **Стоимость на запрос** | Низкая (индекс уже построен) | Средняя (зависит от размера модели) | Выше, чем у LLM (из-за дополнительного поиска), но дешевле fine‑tuning |

Из таблицы видно, что RAG занимает «золотую середину»: он даёт связные, обоснованные ответы с возможностью проверки источников и обновления знаний без переобучения модели.

### 1.2. Проблемы, решаемые RAG

RAG был разработан для преодоления фундаментальных ограничений как классических поисковых систем, так и автономных языковых моделей. Рассмотрим шесть ключевых проблем, которые эффективно решает RAG, и приведём реальные примеры.

**1. Актуальность знаний.** LLM имеют «срез» знаний на момент окончания обучения. Например, GPT‑4 (сентябрь 2021) не знает о событиях после этой даты. Если пользователь спросит: «Какие изменения в налоговом кодексе РФ вступили в силу с 2025 года?», LLM либо даст устаревшую информацию, либо признается в незнании. RAG же может обратиться к актуальной базе законодательных документов, найти свежий текст поправок и сгенерировать ответ, основанный на них.

*Реальный пример:* юридическая компания внедряет RAG-систему, которая ежедневно индексирует новые судебные решения и законы. Адвокат задаёт вопрос о недавнем прецеденте – система находит нужное дело и выдаёт краткое резюме со ссылкой на источник.

**2. Приватность и безопасность данных.** Многие организации не могут отправлять свои внутренние документы в облачные API (из-за GDPR, коммерческой тайны или политик безопасности). Традиционный подход fine‑tuning требует передачи данных поставщику модели или их размещения на собственных GPU, что дорого и не всегда возможно. RAG позволяет хранить все документы локально (в векторной базе внутри корпоративного контура) и использовать локальную LLM (например, через Ollama или vLLM), что гарантирует, что данные никогда не покидают защищённую среду.

*Пример:* банк использует RAG для ответов на вопросы сотрудников о внутренних регламентах. Все документы хранятся на серверах банка, модель также запускается локально. Сотрудник спрашивает: «Какие лимиты на переводы для VIP-клиентов?» – система находит актуальный регламент и генерирует ответ, не раскрывая данные вовне.

**3. Специализированные домены.** В таких областях, как медицина, юриспруденция, техническая документация, LLM часто не имеет достаточной глубины знаний или оперирует общими сведениями. Например, редкое генетическое заболевание может быть описано в единичных клинических рекомендациях, которые не попали в обучающий корпус модели. RAG позволяет подключить специализированную базу (например, PubMed, коллекцию клинических протоколов) и получать точные ответы, основанные на экспертных источниках.

*Пример:* врач-онколог запрашивает рекомендации по лечению редкой саркомы. RAG ищет в базе клинических исследований за последние 5 лет, находит несколько подходящих статей и выдаёт обобщённые рекомендации с цитированием первоисточников.

**4. Прозрачность и проверяемость.** Одна из главных проблем LLM – «чёрный ящик»: невозможно узнать, на основании каких фактов модель выдала ответ. RAG решает эту проблему, предоставляя возможность показать пользователю, из каких конкретно документов была извлечена информация. Это повышает доверие, особенно в критических областях (медицина, юриспруденция, финансы).

*Пример:* студент пишет курсовую работу по истории и спрашивает: «Каковы причины Февральской революции 1917 года?» RAG находит несколько академических монографий, выдаёт ответ и в конце указывает: «Источники: Иванов А.А. 'Причины Февральской революции', 2010, стр. 45–47; Петров Б.В. 'Россия в 1917 году', 2015, стр. 112–115». Студент может проверить первоисточники.

**5. Экономическая эффективность.** Полная тонкая настройка (full fine‑tuning) модели, особенно большой (70B+), требует десятков и сотен тысяч долларов на вычислительные ресурсы и специалистов. RAG же требует лишь однократной индексации документов (что сравнительно дёшево) и выполнение поиска на каждый запрос (операция с вычислительной сложностью O(log N) для ANN-индексов). По оценкам, стоимость одного RAG-запроса на 10–100 раз дешевле, чем обучение модели на новых данных.

*Пример:* стартап создаёт чат-бота для поддержки клиентов интернет-магазина. Вместо того чтобы дообучать модель на сотнях тысяч тикетов, они индексируют базу знаний (статьи, инструкции, политики) и используют RAG. Это позволяет быстро запустить систему и легко обновлять знания при появлении новых товаров или правил, без затрат на переобучение.

**6. Динамическое обновление знаний.** В мире, где информация меняется ежедневно, поддерживать актуальность модели через переобучение практически невозможно. RAG позволяет обновлять знания простым добавлением, удалением или изменением документов в базе. Например, если вышла новая версия технического регламента, достаточно загрузить новый PDF-файл в систему, и уже через несколько минут модель сможет отвечать на вопросы по нему.

*Пример:* производитель программного обеспечения выпускает еженедельные патчи и обновления документации. Техподдержка использует RAG-систему: инженеры добавляют новые релиз-ноуты в базу, и чат-бот сразу начинает давать корректные ответы о новых функциях и исправленных багах, без какого-либо переобучения.

### 1.3. Основные сценарии применения RAG

RAG находит применение в самых разных отраслях. Рассмотрим шесть наиболее распространённых сценариев с пояснением, какие именно проблемы RAG решает в каждом случае.

**1. Корпоративные системы поиска по внутренней документации.** Крупные компании имеют тысячи страниц политик, инструкций, технической документации, отчётов. Сотрудникам часто трудно найти нужную информацию, даже при наличии поиска. RAG позволяет задать вопрос на естественном языке и получить точный ответ с указанием источника. *Решаемые проблемы:* специализированный домен, приватность, прозрачность.

**2. Интеллектуальные FAQ и чат‑боты для поддержки клиентов.** Традиционные FAQ статичны и не покрывают все возможные вопросы. Чат-боты на основе LLM часто галлюцинируют, если не имеют доступа к актуальной базе знаний. RAG-чат-бот подключается к базе статей поддержки, форумов, руководств и выдаёт точные ответы, снижая нагрузку на операторов. *Решаемые проблемы:* актуальность знаний, экономическая эффективность, динамическое обновление.

**3. Юридические ассистенты.** Юристы и адвокаты работают с огромным объёмом законов, постановлений, судебных прецедентов. RAG позволяет быстро находить релевантные нормы и прецеденты, формулировать правовые заключения, проверять согласованность документов. *Решаемые проблемы:* специализированный домен, прозрачность, актуальность.

**4. Медицинские ассистенты.** Врачи могут использовать RAG для поиска по клиническим рекомендациям, фармакологическим справочникам, результатам клинических испытаний. Система помогает быстро принимать решения, особенно в редких или сложных случаях. *Решаемые проблемы:* специализированный домен, проверяемость, актуальность (если база регулярно обновляется).

**5. Образовательные платформы.** Студенты и преподаватели могут задавать вопросы по учебникам, лекциям, научным статьям. RAG не только даёт ответ, но и указывает, где в учебнике эта тема освещена, что способствует самостоятельному изучению. *Решаемые проблемы:* прозрачность, динамическое обновление (при добавлении новых материалов).

**6. Научно-исследовательские системы.** Исследователи тратят до 40% времени на обзор литературы. RAG-система может помочь найти релевантные статьи, сравнить результаты экспериментов, выделить ключевые методы. Интеграция с arXiv, PubMed и другими базами делает этот процесс значительно эффективнее. *Решаемые проблемы:* специализированный домен, актуальность, прозрачность.

### 1.4. Ключевые преимущества RAG перед альтернативами

Подход RAG имеет ряд существенных преимуществ, подтверждённых как теоретическими исследованиями, так и практическим опытом.

**1. Снижение галлюцинаций на 50–70%.** В исследованиях (например, работах по Self‑RAG и CRAG) показано, что использование внешнего контекста уменьшает вероятность фактологических ошибок. Например, в бенчмарке *Natural Questions* RAG-модели достигают точности на 15–20% выше, чем чистая LLM. При этом генерация с привязкой к документам даёт меньше вымысла. Цифра 50–70% — это усреднённое улучшение по метрикам faithfulness (верность фактам) в различных экспериментах.

**2. Обновление знаний без переобучения.** В отличие от fine‑tuning, где для обновления знаний нужно заново обучать модель (что требует времени и ресурсов), RAG позволяет просто добавить новые документы в индекс. Это делает систему готовой к работе с новыми данными в течение минут.

**3. Возможность ссылаться на источники – повышение доверия.** Возможность указать, из какого документа взята информация, критична для корпоративных, юридических и медицинских приложений. Пользователь может проверить первоисточник, что значительно повышает доверие к системе и снижает риски неправильных решений.

**4. Гибкость и модульность архитектуры.** RAG построен на заменяемых компонентах: ретривер, модель эмбеддингов, векторная БД, LLM, промпт-инжиниринг – каждый из них можно обновлять или заменять независимо. Например, можно перейти с open‑source модели на GPT‑4, не меняя остальную систему, или заменить Chroma на Milvus для масштабирования.

**5. Экономия вычислительных ресурсов.** Обучение или дообучение большой модели требует тысяч GPU‑часов и стоит десятки тысяч долларов. RAG же требует только однократной индексации документов (которая выполняется на CPU) и лёгких поисковых запросов. Стоимость одного RAG-запроса (с учётом поиска и генерации) на порядок ниже, чем у fine‑tuned модели.

### 1.5. Визуализация: архитектура RAG и сравнение с альтернативами

**Схема «RAG vs LLM vs Search» (текстовое описание):**

- **Поиск (Search):** Пользователь → запрос → поисковый движок → список документов/ссылок → пользователь (без генерации связного ответа).
- **Чистая LLM:** Пользователь → запрос → LLM (генерирует из весов) → ответ (без ссылок, возможны галлюцинации).
- **RAG:** Пользователь → запрос → ретривер → поиск в векторной БД → контекст (документы) → формирование промпта (контекст + запрос) → LLM → ответ со ссылками → пользователь.

Ниже представлена диаграмма в формате Mermaid, показывающая взаимодействие компонентов как для этапа индексации (offline), так и для этапа инференса (online).

```mermaid
flowchart TD
    subgraph Offline["Этап индексации (offline)"]
        A[Документы] --> B[Извлечение текста и очистка]
        B --> C[Разбиение на чанки]
        C --> D[Генерация эмбеддингов]
        D --> E[Сохранение в векторную БД]
        E --> F[(Векторная база данных)]
    end

    subgraph Online["Этап инференса (online)"]
        Q[Запрос пользователя] --> G[Генерация эмбеддинга запроса]
        G --> H[Поиск ближайших соседей в векторной БД]
        H --> I[Извлечение топ-k чанков]
        I --> J[Переранжирование / фильтрация]
        J --> K[Формирование промпта с контекстом]
        K --> L[LLM генерация ответа]
        L --> M[Постобработка / добавление ссылок]
        M --> R[Ответ пользователю]
    end

    F --> H
    style F fill:#f9f,stroke:#333,stroke-width:2px
```

На схеме видно, что этап индексации и этап инференса разделены: индексация выполняется один раз (или периодически), а инференс — каждый запрос.

### 1.6. Сквозные примеры работы RAG

**Пример 1. Налоговый кодекс.** Пользователь (бухгалтер) спрашивает: *«Какие изменения по налогу на прибыль вступили в силу с января 2025 года?»*. Система RAG:
1. Преобразует запрос в эмбеддинг.
2. Ищет в векторной БД, содержащей все актуальные законодательные документы (например, тексты законов, постановлений, разъяснений Минфина).
3. Находит несколько релевантных фрагментов: текст поправок, даты вступления, комментарии.
4. Формирует промпт: *«Контекст: [фрагмент 1], [фрагмент 2]. Вопрос: ...»*.
5. LLM генерирует ответ: *«С 1 января 2025 года ставка налога на прибыль для ИТ-компаний снижена с 20% до 17%. Также изменён порядок расчёта амортизации по нематериальным активам (см. ст. 259.3 НК РФ). Источник: Федеральный закон № 123-ФЗ от 20.12.2024, ст. 1, п. 3»*.
В ответе указана конкретная статья и документ, что позволяет бухгалтеру проверить информацию.

**Пример 2. Редкое заболевание.** Врач-онколог спрашивает: *«Какие схемы лечения саркомы Юинга рекомендуются для детей младше 5 лет?»*. Система RAG:
1. Ищет в базе клинических рекомендаций (например, NCCN, PubMed, локальные протоколы).
2. Находит несколько релевантных статей и клинических случаев.
3. Формирует промпт с контекстом из 3–5 источников.
4. LLM генерирует ответ: *«Согласно клиническим рекомендациям NCCN 2024, для детей до 5 лет предпочтительна схема VAC (винкристин, актиномицин, циклофосфамид) с уменьшенными дозировками. Также рекомендуется проведение локальной лучевой терапии после 3 циклов химиотерапии (источники: NCCN Guidelines, версия 2.2024, стр. 12; статья Smith et al., JCO 2023)»*.
Врач получает конкретные рекомендации с источниками, что повышает уверенность в правильности решения.

### 1.7. Математические основы (интуиция)

На данном этапе мы не будем углубляться в детали, но дадим интуитивное понимание ключевой математической операции RAG — оценки релевантности.

В основе векторного поиска лежит преобразование текста в векторы фиксированной размерности (эмбеддинги) с помощью нейросетевых моделей (например, *sentence‑transformers*). Затем релевантность между запросом и документом измеряется через **косинусное расстояние** (или косинусную близость):

$$
\text{cosine\_similarity}(q, d) = \frac{\mathbf{q} \cdot \mathbf{d}}{\|\mathbf{q}\| \cdot \|\mathbf{d}\|}
$$

где $\mathbf{q}$ — вектор запроса, $\mathbf{d}$ — вектор документа. Значение близости лежит в диапазоне $[-1, 1]$, причём значение, близкое к 1, означает высокую релевантность (векторы сонаправлены). Поиск ближайших соседей выполняется с использованием приближённых алгоритмов (ANN), таких как HNSW или IVF, чтобы обеспечить миллисекундную задержку даже для миллиардов документов. Более подробно математика будет рассмотрена в следующих темах.

### 1.8. Контрольные вопросы и задания

**Вопросы для самопроверки (с ответами):**

1. *Чем RAG отличается от тонкой настройки (fine‑tuning) языковой модели?*  
   **Ответ:** RAG не изменяет веса модели, а использует внешний поиск для предоставления актуального контекста на каждый запрос. Fine‑tuning изменяет веса модели на основе размеченных данных, что требует значительных вычислительных ресурсов и переобучения при обновлении знаний. RAG дешевле, проще в обновлении и обеспечивает прозрачность источников.

2. *Какие проблемы решает RAG, которые не решаются классическим поиском?*  
   **Ответ:** Классический поиск возвращает документы, но не генерирует связный ответ, не может переформулировать информацию и не адаптируется к контексту диалога. RAG же генерирует структурированный, понятный ответ, извлекает ключевую информацию из нескольких источников и может указывать ссылки.

3. *Почему RAG считается более экономически эффективным, чем fine‑tuning?*  
   **Ответ:** Fine‑tuning требует дорогостоящих GPU-часов (например, дообучение модели 70B может стоить >100 000 $). RAG требует лишь однократной индексации документов (выполняется на CPU) и лёгких поисковых операций на каждый запрос, что на несколько порядков дешевле при большом количестве запросов.

**Практические задания:**

1. Найдите в интернете не менее трёх примеров реального внедрения RAG в компаниях (можно поискать кейсы на сайтах Pinecone, Weaviate, Qdrant, а также в блогах по AI). Опишите каждое внедрение: какая задача решалась, какие компоненты использовались (ретривер, векторная БД, LLM), какие результаты были достигнуты. Сделайте краткий обзор (1–2 страницы).

2. Напишите эссе (объём 1–2 страницы) на тему: *«Почему в 2024–2026 годах RAG стал более популярным подходом, чем тонкая настройка моделей?»*. В эссе обязательно затроньте: а) рост популярности векторных баз данных; б) появление качественных open‑source LLM; в) требования к приватности и актуальности знаний; г) экономические факторы; д) сравнение с другими методами (fine‑tuning, промпт-инжиниринг). Аргументируйте свою точку зрения.

### 1.9. Список литературы для углублённого изучения

1. **Lewis, P., Perez, E., Piktus, A., et al. (2020).** *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*. – arXiv:2005.11401. Оригинальная статья, заложившая основы RAG.

2. **Gao, Y., Xiong, Y., Gao, X., et al. (2023).** *Retrieval-Augmented Generation for Large Language Models: A Survey*. – arXiv:2312.10997. Обзор современных подходов к RAG, включая продвинутые архитектуры.

3. **Asai, A., et al. (2023).** *Self-RAG: Learning to Retrieve, Generate, and Critique through Self-Reflection*. – ICLR 2024. Статья о Self‑RAG, одном из самых влиятельных расширений RAG.

4. **Yan, S., et al. (2024).** *Corrective Retrieval Augmented Generation*. – Работа по CRAG, описывающая механизмы исправления ошибок поиска.

5. **Hugging Face Blog.** *Retrieval-Augmented Generation (RAG) – A Comprehensive Guide*. – https://huggingface.co/blog/rag. Практическое руководство по реализации RAG с открытым кодом.

6. **Pinecone Documentation.** *RAG with LLMs*. – https://docs.pinecone.io/docs/rag. Описание практических подходов с использованием Pinecone.

7. **Weaviate Blog.** *The Complete Guide to Retrieval-Augmented Generation*. – https://weaviate.io/blog. Цикл статей с примерами кода и архитектурными решениями.




## Тема 2. Архитектура RAG-системы

После того как мы определили, что такое RAG и какие задачи он решает, необходимо погрузиться в его внутреннее устройство. Понимание архитектуры — ключ к грамотному проектированию, оптимизации и отладке систем, работающих в реальных условиях. В этой теме мы детально разберём все компоненты RAG, этапы его работы, основные типы архитектур, сравним RAG с альтернативными подходами и дадим практические рекомендации по выбору.

---

### 2.1. Основные компоненты архитектуры

Любая RAG-система состоит из пяти функциональных модулей. Их можно представить как последовательный конвейер, в котором каждый модуль выполняет строго определённую задачу, а также как набор независимых блоков, которые можно заменять, улучшать или масштабировать.

**1. Модуль индексации (Ingestion).** Это «фабрика знаний» системы. Он отвечает за подготовку документов к поиску и выполняется в фоновом режиме (offline). Основные шаги:

- **Загрузка** — получение документов из различных источников (локальные папки, базы данных, веб-API, облачные хранилища). Инструменты: `pypdf`, `python-docx`, `beautifulsoup4`, `pandas`, а также фреймворки `LangChain` и `LlamaIndex`, предоставляющие унифицированные загрузчики (`DirectoryLoader`, `WebBaseLoader`).
- **Предобработка и очистка** — удаление шума, нормализация текста, исправление кодировок, извлечение основного содержимого (без рекламы, навигации и т.д.). Часто применяются библиотеки `re`, `ftfy`, `textwrap`, а для работы с HTML — `readability-lxml`.
- **Чанкинг (разбиение на фрагменты)** — деление длинных документов на смысловые блоки фиксированного размера или по предложениям/абзацам. От размера чанка зависит качество поиска: слишком маленький чанк теряет контекст, слишком большой — вносит шум. Инструменты: `RecursiveCharacterTextSplitter` (LangChain), `SentenceTransformersTokenTextSplitter`.
- **Генерация эмбеддингов** — преобразование текстовых чанков в числовые векторы с помощью предобученных моделей эмбеддингов. Популярные модели: `all‑MiniLM‑L6‑v2` (быстрая, 384d), `BAAI/bge‑base‑en‑v1.5` (высокое качество, 768d), `intfloat/multilingual‑e5‑large` (многоязычная, 1024d). Реализация: `sentence‑transformers`, `HuggingFace Embedding` классы.
- **Сохранение в векторной базе данных** — запись полученных векторов вместе с метаданными (имя документа, номер страницы, дата и т.д.) в специализированное хранилище. Популярные ВБД: `Chroma` (для прототипов), `FAISS` (высокопроизводительная библиотека, но без встроенного хранения метаданных), `Qdrant`, `Weaviate`, `Milvus`, `Pinecone`.

**2. Модуль поиска (Retriever).** Это «глаза» системы. Он получает запрос пользователя и находит наиболее релевантные фрагменты в индексированном корпусе. Поиск может быть:

- **Векторный (семантический)** — использует эмбеддинги запроса и документов, вычисляет косинусную близость. Работает хорошо для запросов, сформулированных на естественном языке, и улавливает смысловые нюансы.
- **Лексический (на основе ключевых слов)** — использует инвертированные индексы и алгоритмы типа BM25 (Okapi BM25) или TF‑IDF. Хорош для точных совпадений, кодов, номеров, имён собственных, но игнорирует семантику.
- **Гибридный** — комбинирует результаты векторного и лексического поиска (например, взвешенная сумма баллов или реранкинг). Даёт наилучшее качество в широком спектре запросов.

Инструменты: для векторного поиска — `FAISS`, `Chroma`; для лексического — `rank_bm25`, `elasticsearch`; для гибридного — `Weaviate` (с поддержкой BM25 + векторного), а также ручная реализация с объединением результатов.

**3. Модуль генерации (Generator).** Это «голос» системы. Он принимает на вход запрос и контекст (найденные документы) и генерирует связный ответ. Генератор — это языковая модель (LLM), которая может быть:

- **Локальной** — запущенной на собственных серверах (например, `Qwen2.5‑7B`, `Llama‑3.2‑3B`, `Mistral‑7B` через `Ollama`, `vLLM`, `llama.cpp`). Обеспечивает приватность, но требует вычислительных ресурсов.
- **Облачной** — доступ к API от OpenAI (GPT‑4, GPT‑4o), Anthropic (Claude), Cohere (Command), Google (Gemini). Даёт высокое качество, но платное и требует передачи данных.

Важно, чтобы модель имела достаточное контекстное окно для размещения всех релевантных чанков (обычно 4–8 чанков по 256–512 токенов).

**4. Модуль интеграции (Integration).** Это «мозг», связывающий поиск и генерацию. Он включает:

- **Формирование промпта** — создание текста, который подаётся на вход LLM. Промпт обычно содержит системную инструкцию (задающую роль модели), контекст (найденные чанки с возможными ссылками), и сам вопрос пользователя. Важно правильно структурировать контекст, чтобы модель не перепутала источник и не начала галлюцинировать.
- **Управление длиной контекста** — если общая длина чанков превышает допустимый лимит, применяется усечение (например, оставляем только топ‑K самых релевантных) или суммаризация.
- **Постобработка ответа** — добавление ссылок на источники, проверка согласованности, форматирование (Markdown, таблицы, списки).

Инструменты: ручное шаблонирование с `f‑strings` или использование `LangChain` (класс `PromptTemplate`), `ChatPromptTemplate`.

**5. Модуль обратной связи (Feedback).** Это система сбора данных для непрерывного улучшения. Он включает:

- **Логирование** всех запросов, найденных документов, сгенерированных ответов, времён выполнения.
- **Сбор метрик** качества — автоматических (BLEU, ROUGE, faithfulness) и пользовательских (лайки/дизлайки, оценки). Инструменты: `RAGAS`, `DeepEval`, `TruLens`.
- **Анализ ошибок** — выявление частых проблем (например, частые случаи «не найдено»), чтобы скорректировать чанкинг или поиск.
- **Итеративное улучшение** — обновление индекса, переобучение модели эмбеддингов, настройка параметров реранкинга на основе собранной обратной связи.

---

### 2.2. Этапы работы RAG (offline и online)

Процесс функционирования RAG-системы чётко делится на две фазы: подготовительную (индексацию) и исполнительную (инференс). Понимание каждого этапа важно для оптимизации.

#### Offline (индексация)

Выполняется один раз (или периодически) перед началом работы системы.

| Шаг | Описание | Ключевые параметры, влияющие на качество/скорость |
| :--- | :--- | :--- |
| **1. Загрузка документов** | Сбор всех исходных документов из различных источников. | Формат, объём, необходимость обновления. |
| **2. Предобработка** | Очистка, извлечение основного текста, удаление шумов. | Качество парсинга (важно для PDF и HTML). |
| **3. Чанкинг** | Разбиение текста на фрагменты (например, по 500 токенов с overlap 50). | `chunk_size`, `chunk_overlap`, стратегия (recursive, semantic). Влияют на recall и precision. |
| **4. Генерация эмбеддингов** | Преобразование каждого чанка в вектор фиксированной размерности. | Модель эмбеддингов, размерность, использование GPU/CPU. |
| **5. Построение индекса** | Сохранение векторов и метаданных в ВБД с построением индекса (HNSW, IVF). | Тип индекса, параметры (M, ef_construction для HNSW), влияют на скорость поиска и точность. |

#### Online (инференс)

Выполняется для каждого пользовательского запроса.

| Шаг | Описание | Ключевые параметры, влияющие на качество/скорость |
| :--- | :--- | :--- |
| **1. Запрос пользователя** | Пользователь вводит вопрос или сообщение. | Язык, длина, сложность. |
| **2. (Опционально) Переформулировка** | Улучшение запроса (например, добавление синонимов, уточнение местоимений). | Модель для переформулировки, качество исходного вопроса. |
| **3. Поиск (Retrieval)** | Поиск по индексу: вычисляется эмбеддинг запроса, выполняется ANN-поиск. | `top_k` (число возвращаемых чанков), тип индекса, фильтрация по метаданным. |
| **4. Реранкинг (опционально)** | Уточнение порядка найденных чанков с помощью cross‑encoder. | Модель cross‑encoder, количество чанков для реранкинга. |
| **5. Формирование промпта** | Сборка системной инструкции, контекста из релевантных чанков и вопроса пользователя. | Шаблон промпта, максимальная длина контекста. |
| **6. Генерация (LLM)** | Подача промпта в LLM и получение ответа. | Модель LLM, параметры генерации (temperature, top_p, max_tokens). |
| **7. Постобработка** | Добавление ссылок на источники, форматирование, проверка фактов. | Правила добавления ссылок, шаблоны вывода. |

---

### 2.3. Типы RAG-систем

Архитектуры RAG можно классифицировать по уровню сложности и функциональности.

**Naive RAG** — это базовый вариант, в котором есть ровно один шаг поиска и один шаг генерации. Нет переранжирования, гибридного поиска, фильтрации. Подходит для простых FAQ, где документы хорошо структурированы, а запросы типовые. Недостатки: может возвращать нерелевантные чанки, не учитывает синонимы, не использует сложную логику.

**Advanced RAG** — включает улучшения:
- **Гибридный поиск** — комбинация векторного и BM25 для учета и семантики, и точных совпадений.
- **Реранкинг** — применение cross‑encoder для уточнения порядка чанков.
- **Фильтрация по метаданным** — например, только документы за последний год, или только от определённого автора.
- **Адаптивный чанкинг** — например, семантическое разбиение.
Такой подход даёт значительный прирост качества (на 5–15% по метрикам) ценой небольшого увеличения времени ответа.

**Modular RAG** — архитектура, где каждый компонент (парсер, сплиттер, модель эмбеддингов, ВБД, реранкер, LLM) является независимым модулем с чётким интерфейсом. Модули можно заменять, комбинировать, переиспользовать в разных проектах. Это самый гибкий подход, рекомендуемый для продакшена, особенно когда система будет развиваться.

**Гибридный поиск** часто выделяют как отдельную разновидность, так как сочетание BM25 и векторного поиска — один из самых эффективных способов увеличить recall. Реализуется либо через объединение ранжированных списков, либо через использование специальных ВБД с поддержкой гибридных запросов (например, Weaviate).

**Agentic RAG** — следующий уровень эволюции, где LLM выступает в роли агента, который сам принимает решения: когда искать, что именно искать, в каком порядке выполнять операции. Агент может разбить сложный запрос на несколько подзапросов, выполнять поиск в разных базах, проверять полученные факты и, если нужно, уточнять запрос. Этот подход реализуется с использованием фреймворков типа `LangGraph` или `AutoGen` и позволяет решать самые сложные задачи, но требует больше вычислительных ресурсов и тщательной отладки.

**Сравнительная таблица типов RAG-систем**

| Критерий | Naive RAG | Advanced RAG | Modular RAG | Agentic RAG |
| :--- | :--- | :--- | :--- | :--- |
| **Сложность реализации** | Низкая | Средняя | Выше средней | Высокая |
| **Качество ответа** | Среднее (зависит от данных) | Хорошее (выше на 10–20%) | Очень хорошее (гибкость) | Потенциально наилучшее (адаптивность) |
| **Время ответа (латенси)** | Низкое | Умеренное (реранкинг + гибрид) | Умеренное | Может быть высоким (много шагов) |
| **Гибкость** | Низкая (всё зафиксировано) | Средняя | Высокая (замена модулей) | Очень высокая (решения агента) |
| **Стоимость (на запрос)** | Низкая | Средняя | Средняя | Высокая (больше вызовов LLM) |
| **Обновляемость** | Простая (обновление индекса) | Простая | Простая | Сложная (логика агента) |

---

### 2.4. Сравнение RAG с другими подходами

Чтобы правильно выбрать технологию для своей задачи, полезно сравнить RAG с альтернативными методами.

**RAG vs Fine‑tuning (дообучение).** Fine‑tuning изменяет веса модели под конкретную задачу, что требует дорогих GPU-ресурсов и размеченных данных. Оно даёт хорошее качество, если задача требует изменения стиля или манеры ответа (например, моделирование личности). Однако fine‑tuning «замораживает» знания: для обновления информации нужно переобучать модель. RAG же позволяет использовать свежие данные без переобучения, дешевле и прозрачнее. **Рекомендация:** для фактологических вопросов → RAG; для изменения стиля/формата → fine‑tuning; в идеале — комбинация (fine‑tuning для стиля + RAG для фактов).

**RAG vs Prompt Engineering.** Промпт-инжиниринг (разработка шаблонов вопросов, инструкций) — это бесплатный способ улучшить ответы LLM, но он не добавляет модели новых знаний. Промпт только меняет форму ответа или способ рассуждения. RAG же снабжает модель конкретными фактами. Промпт-инжиниринг всегда используется вместе с RAG (для правильного оформления контекста), но сам по себе не решает проблему актуальности знаний.

**RAG vs Semantic Search (семантический поиск).** Семантический поиск возвращает список документов, отсортированных по релевантности. Это полезно для исследователей, но не даёт готового ответа. RAG идёт дальше, используя документы как контекст для генерации связного ответа. Это особенно важно в диалоговых системах и для пользователей, которые не хотят читать много документов.

**Рекомендации по выбору:**

| Тип задачи | Рекомендуемый подход |
| :--- | :--- |
| Ответы на фактологические вопросы, работа с документами, поддержка клиентов | **RAG** |
| Требуется изменить стиль общения или жанр (формальный → неформальный) | **Fine‑tuning** (или комбинация) |
| Нет данных, но нужно улучшить логику ответов | **Prompt Engineering** |
| Пользователь хочет сам изучить документы | **Semantic Search** |
| Сложные, многошаговые задачи с необходимостью принятия решений | **Agentic RAG** |

---

### 2.5. Визуализация архитектуры

#### Диаграмма потока данных (offline/online) в Mermaid

```mermaid
flowchart TD
    subgraph Offline["Offline: Индексация"]
        A[Документы] --> B[Очистка и извлечение текста]
        B --> C[Чанкинг]
        C --> D[Генерация эмбеддингов]
        D --> E[Сохранение в векторную БД]
        E --> F[(Векторная БД с индексом)]
    end

    subgraph Online["Online: Инференс"]
        G[Запрос пользователя] --> H[Переформулировка запроса (опционально)]
        H --> I[Вычисление эмбеддинга запроса]
        I --> J[Поиск в векторной БД]
        J --> K[Реранкинг (опционально)]
        K --> L[Формирование промпта\n(система + контекст + вопрос)]
        L --> M[LLM генерация]
        M --> N[Постобработка\n(добавление ссылок, формат)]
        N --> O[Ответ пользователю]
    end

    F --> J
    style F fill:#f9f,stroke:#333,stroke-width:2px
```

#### Схема архитектуры с компонентами

Ниже представлена схема, показывающая взаимодействие компонентов в RAG-системе.

```mermaid
flowchart LR
    subgraph Ingestion["Модуль индексации"]
        direction LR
        P[Parser] --> S[Splitter]
        S --> E[Embedder]
        E --> V[Vector DB]
    end

    subgraph Retrieval["Модуль поиска"]
        direction TB
        Q[Запрос] --> QE[Embedder]
        QE --> VS[Векторный поиск]
        VS --> RR[Реранкер] --> R[Результаты]
    end

    subgraph Generation["Модуль генерации"]
        direction TB
        R --> PF[Промпт-инжиниринг]
        PF --> LLM[LLM] --> Post[Постобработка]
    end

    subgraph Feedback["Модуль обратной связи"]
        direction LR
        Metrics[Сбор метрик] --> Logs[Логирование]
        Logs --> Analysis[Анализ] --> Update[Обновление параметров]
    end

    Ingestion --> Retrieval
    Retrieval --> Generation
    Generation --> Feedback
    Feedback --> Ingestion
    Feedback --> Retrieval
    Feedback --> Generation
```

---

### 2.6. Математические основы

В основе поиска лежат два ключевых математических механизма: **косинусная близость** (для оценки релевантности) и **приближённый поиск ближайших соседей (ANN)** для обеспечения скорости.

**Косинусная близость** измеряет угол между вектором запроса $\mathbf{q}$ и вектором документа $\mathbf{d}$:

$$
\text{cosine\_similarity}(\mathbf{q}, \mathbf{d}) = \frac{\mathbf{q} \cdot \mathbf{d}}{\|\mathbf{q}\| \cdot \|\mathbf{d}\|} = \frac{\sum_{i=1}^{n} q_i d_i}{\sqrt{\sum_{i=1}^{n} q_i^2} \sqrt{\sum_{i=1}^{n} d_i^2}}
$$

Эта метрика не зависит от длины векторов (они нормализованы) и даёт значения в диапазоне $[-1, 1]$, где 1 означает, что векторы коллинеарны (максимально схожи), 0 — ортогональны (независимы), -1 — противоположно направлены. В реальных задачах близость > 0.7 обычно считается хорошим совпадением. Косинусное расстояние (1 - косинусная близость) часто используется как метрика расстояния.

**Приближённый поиск ближайших соседей (ANN)** решает проблему линейного сканирования всех векторов (что для миллиардов документов невозможно). Два наиболее популярных алгоритма:

- **HNSW (Hierarchical Navigable Small World)** — строит многослойную графовую структуру, где каждый слой — это граф малого мира с убывающей плотностью. Поиск начинается с верхнего (самого разреженного) слоя и спускается вниз, каждый раз находя ближайшие узлы. Параметры: `M` (количество связей на узел), `ef_construction` (размер динамического списка при построении). HNSW обеспечивает высокую точность и скорость, но требует больше памяти.

- **IVF (Inverted File Index)** — разбивает всё множество векторов на кластеры (с помощью k-means) и строит инвертированный индекс: для каждого кластера хранятся идентификаторы векторов, принадлежащих ему. При поиске сначала определяются ближайшие кластеры (обычно `nprobe`), а затем сканируются только векторы внутри этих кластеров. Это значительно сокращает количество вычислений. Часто комбинируется с Product Quantization (PQ) для сжатия векторов.

Компромисс между точностью (recall) и скоростью достигается подбором параметров (для HNSW — `M` и `ef_search`; для IVF — `nprobe`). В большинстве систем удаётся достичь recall > 95% при задержке менее 100 мс на миллион векторов.

---

### 2.7. Сквозной пример работы системы

Рассмотрим запрос пользователя: *«Какие налоги нужно платить самозанятому в 2025 году?»* в системе RAG для бухгалтерского консалтинга.

1. **Модуль интеграции** получает запрос и запускает пайплайн.
2. **Модуль поиска**:
   - Вычисляет эмбеддинг запроса (например, с помощью `BAAI/bge‑base‑en‑v1.5`).
   - Выполняет поиск по векторной БД, содержащей все законы, постановления, разъяснения Минфина за последние 3 года. Возвращает топ‑5 чанков с косинусной близостью: 0.92, 0.87, 0.81, 0.75, 0.68.
   - Применяет реранкинг с cross‑encoder, уточняя порядок: теперь топ‑3 чанка имеют близость 0.95, 0.93, 0.88.
3. **Модуль интеграции**:
   - Формирует промпт: *«Ты — налоговый консультант. Контекст: [чанк 1: текст ФЗ № 54, ст. 3], [чанк 2: разъяснение Минфина от 10.12.2024], [чанк 3: пример расчёта налога]. Вопрос пользователя: Какие налоги нужно платить самозанятому в 2025 году?»*.
   - Проверяет, что общая длина контекста влезает в окно LLM.
4. **Модуль генерации** (локальная LLM, Qwen2.5‑7B) генерирует ответ:
   *«В 2025 году самозанятые обязаны уплачивать налог на профессиональный доход (НПД) по ставке 4% (при работе с физлицами) или 6% (с юрлицами). Также с 2025 года введён обязательный взнос на медицинское страхование в размере 1000 рублей в квартал (ФЗ № 54, ст. 3). Сроки уплаты: до 25 числа месяца, следующего за отчётным кварталом. Источники: разъяснение Минфина от 10.12.2024, пример расчёта в постановлении № 123.»*
5. **Постобработка** добавляет ссылки на источники в формате `[1]`, `[2]`, `[3]` в конце ответа.

Система выдаёт ответ за 2.3 секунды, все источники корректно проставлены.

---

### 2.8. Контрольные вопросы и задания

**Вопросы для самопроверки:**

1. В чём разница между этапами индексации и инференса в RAG? Почему их необходимо разделять?  
   *Ответ: Индексация выполняется один раз (или периодически) для подготовки данных, требует больших вычислительных ресурсов, но не критична по времени. Инференс выполняется для каждого запроса и должен быть быстрым. Разделение позволяет оптимизировать каждый этап независимо.*

2. Какие преимущества даёт гибридный поиск (BM25 + векторный) по сравнению с чистым векторным поиском?  
   *Ответ: Гибридный поиск учитывает как семантическую близость (векторы), так и точные лексические совпадения (BM25), что особенно полезно для запросов с именами, кодами, номерами или специфической терминологией. Это повышает recall (полноту) и precision (точность).*

3. В каких случаях Agentic RAG предпочтительнее Advanced RAG?  
   *Ответ: Когда запросы сложные, многокомпонентные (например, «сравните показатели компании А и компании Б за последние 5 лет, выделив основные тренды»), требуют нескольких шагов поиска, проверки фактов или обращения к нескольким источникам. Agentic RAG может самостоятельно планировать последовательность действий, что даёт более качественный результат, хотя и за большее время.*

**Практические задания:**

1. **Постройте схему выбора архитектуры RAG** в зависимости от требований проекта. Используйте бинарные критерии: нужна ли высокая точность (>95%), допустимая задержка (менее 1 сек vs 3–5 сек), бюджет на GPU, частота обновления данных. Оформите в виде блок-схемы (можно текстовой) и опишите, для каких комбинаций выбираете Naive, Advanced, Modular или Agentic RAG.

2. **Опишите план перехода от Naive RAG к Advanced** в существующей системе:
   - Какие именно компоненты нужно добавить или заменить?
   - В каком порядке их внедрять (чтобы сохранить работоспособность)?
   - Оцените, на сколько процентов увеличится время ответа и какие метрики качества улучшатся.
   Ответ должен быть пошаговым, с пояснениями.

---

### 2.9. Список литературы

1. **Lewis, P., Perez, E., Piktus, A., et al. (2020).** *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*. – arXiv:2005.11401.
2. **Gao, Y., Xiong, Y., Gao, X., et al. (2023).** *Retrieval-Augmented Generation for Large Language Models: A Survey*. – arXiv:2312.10997.
3. **LangChain Documentation.** *RAG Concepts*. – https://python.langchain.com/docs/use_cases/rag/ – практическое руководство по компонентам RAG.
4. **Qdrant Blog.** *Vector Search Algorithms: HNSW vs IVF vs PQ*. – https://qdrant.tech/articles/ – разбор математики ANN.
5. **Hugging Face Blog.** *Introduction to Retrieval-Augmented Generation (RAG)*. – https://huggingface.co/blog/rag.
6. **Pinecone Academy.** *RAG Architecture Patterns*. – https://www.pinecone.io/learn/ – описание различных архитектур.
7. **Weaviate Blog.** *Hybrid Search Explained*. – https://weaviate.io/blog/hybrid-search – о комбинировании BM25 и векторного поиска.



## Тема 3. Методы оценки качества RAG (Расширенное теоретико-практическое руководство)

Оценка качества RAG-системы — это многоаспектная задача, требующая раздельного анализа эффективности поиска (retrieval) и качества генерации (generation), а также их взаимодействия. В отличие от классических задач информационного поиска или машинного перевода, здесь нет единственной «истинной» метрики: приходится учитывать точность найденных документов, верность сгенерированного ответа фактам, релевантность ответа вопросу и пользовательское восприятие. В этой теме мы детально разберём математический аппарат, лежащий в основе оценки RAG, а также современные фреймворки и подходы к мониторингу в реальных условиях.

---

### 3.1. Метрики оценки поиска (Retrieval Metrics)

Метрики поиска измеряют, насколько хорошо ретривер ранжирует документы относительно запроса. Они оперируют понятием **релевантности** — бинарной (документ либо релевантен, либо нет) или градуальной (например, оценка от 0 до 3). В большинстве практических задач используют бинарную релевантность для упрощения расчётов.

Введём обозначения:
- $Q$ — множество тестовых запросов.
- $D$ — множество всех документов в корпусе.
- $R(q) \subseteq D$ — множество документов, релевантных запросу $q$.
- $retrieved(q, k)$ — множество из $k$ документов, возвращённых ретривером (топ‑$k$).

#### 3.1.1. Precision@k (Точность среди топ‑k)

**Определение:** доля релевантных документов среди первых $k$ возвращённых.

**Формула:**

$$
\text{Precision@k}(q) = \frac{| \text{retrieved}(q, k) \cap R(q) |}{k}.
$$

**Интерпретация:** показывает, насколько «чист» результат в верхней части выдачи. Значение 0.8 при $k=5$ означает, что из пяти первых документов четыре релевантны. Высокая Precision@k важна, когда пользователь просматривает только первые несколько результатов.

**Агрегация:** обычно вычисляется среднее арифметическое по всем запросам: $\text{Precision@k} = \frac{1}{|Q|} \sum_{q \in Q} \text{Precision@k}(q)$.

#### 3.1.2. Recall@k (Полнота среди топ‑k)

**Определение:** доля всех релевантных документов, которые были найдены среди первых $k$.

**Формула:**

$$
\text{Recall@k}(q) = \frac{| \text{retrieved}(q, k) \cap R(q) |}{| R(q) |}.
$$

**Интерпретация:** показывает, сколько процентов от всех существующих релевантных документов мы смогли покрыть. Recall@5 = 0.6 означает, что мы нашли 60% всех релевантных документов. Эта метрика критична в сценариях, где важно не пропустить ни одного важного документа (например, юридический поиск).

**Замечание:** Recall@k монотонно не убывает с ростом $k$; при $k = |D|$ достигает 1 (если все релевантные документы есть в корпусе). Однако на практике $k$ обычно невелико (5–20), поэтому Recall@k часто остаётся низким, что стимулирует улучшать ретривер.

#### 3.1.3. MRR (Mean Reciprocal Rank)

**Определение:** среднее значение обратного ранга первого релевантного документа.

Для одного запроса $q$:

$$
\text{RR}(q) = \frac{1}{\text{rank}_q},
$$

где $\text{rank}_q$ — позиция первого релевантного документа в выдаче; если ни одного релевантного не найдено, то $\text{RR}(q) = 0$.

Тогда:

$$
\text{MRR} = \frac{1}{|Q|} \sum_{q \in Q} \text{RR}(q).
$$

**Интерпретация:** MRR показывает, насколько высоко в среднем находится самый релевантный документ. Используется в задачах, где пользователю важен только один лучший ответ (например, поиск по FAQ, вопросно-ответные системы). Значение 0.5 означает, что в среднем первый релевантный документ находится на второй позиции (поскольку $\frac{1}{2} = 0.5$).

#### 3.1.4. MAP (Mean Average Precision)

**Определение:** среднее значение средней точности (Average Precision) по всем запросам.

Для одного запроса $q$, у которого есть $m_q$ релевантных документов, расположенных на позициях $p_1 < p_2 < \dots < p_{m_q}$ (где $p_i$ — позиция $i$-го релевантного документа), AP вычисляется как:

$$
\text{AP}(q) = \frac{1}{m_q} \sum_{i=1}^{m_q} \text{Precision@}p_i.
$$

Затем:

$$
\text{MAP} = \frac{1}{|Q|} \sum_{q \in Q} \text{AP}(q).
$$

**Интерпретация:** MAP учитывает порядок всех релевантных документов, штрафуя за низкое расположение каждого из них. Это одна из наиболее информативных метрик для ранжирования, так как она суммирует точность на всех позициях, где встречаются релевантные документы. MAP = 1 достигается, когда все релевантные документы находятся в самом начале выдачи в правильном порядке.

#### 3.1.5. NDCG (Normalized Discounted Cumulative Gain)

**Определение:** метрика, использующая градуальную релевантность (например, оценка от 0 до 3) и дисконтирование по позиции. Она нормализуется на идеальный порядок, чтобы значение лежало в интервале $[0, 1]$.

**Формула:**

Сначала вычисляется DCG@k:

$$
\text{DCG@k}(q) = \sum_{i=1}^{k} \frac{2^{rel_i} - 1}{\log_2(i+1)},
$$

где $rel_i$ — оценка релевантности документа на позиции $i$. Логарифмический знаменатель ($\log_2(i+1)$) обеспечивает дисконтирование: вклад документа падает с ростом его позиции.

Затем IDCG@k — это DCG для идеального упорядочивания, когда документы отсортированы по убыванию релевантности. Наконец:

$$
\text{NDCG@k}(q) = \frac{\text{DCG@k}(q)}{\text{IDCG@k}(q)}.
$$

**Интерпретация:** NDCG учитывает не только наличие релевантных документов, но и их оценку, и штрафует за низкое ранжирование высокорелевантных документов. Значение 1 означает идеальное ранжирование. NDCG является стандартом в задачах ранжирования, таких как поиск в интернете, благодаря своей чувствительности к порядку и градуальности.

---

**Пример расчёта метрик поиска (иллюстрация):**

Пусть корпус состоит из 5 документов: `[d1, d2, d3, d4, d5]`. Для запроса `q1` релевантны `R(q1) = {d2, d4}`. Ретривер вернул ранжированный список: `[d1, d2, d3, d4, d5]`. Рассчитаем метрики для разных k.

| k | retrieved(k) | релевантные среди retrieved | Precision@k | Recall@k |
|---|--------------|------------------------------|-------------|----------|
| 1 | [d1]         | 0                            | 0/1 = 0.0   | 0/2 = 0.0 |
| 2 | [d1, d2]     | {d2} = 1                     | 1/2 = 0.5   | 1/2 = 0.5 |
| 3 | [d1, d2, d3] | {d2} = 1                     | 1/3 ≈ 0.333 | 1/2 = 0.5 |
| 4 | [d1..d4]     | {d2, d4} = 2                 | 2/4 = 0.5   | 2/2 = 1.0 |
| 5 | все          | {d2, d4} = 2                 | 2/5 = 0.4   | 2/2 = 1.0 |

**MRR:** первый релевантный документ (d2) на позиции 2 → RR = 1/2 = 0.5.

**MAP:**  
Precision@1 = 0, Precision@2 = 0.5, Precision@4 = 0.5 → AP = (0 + 0.5 + 0.5) / 2 = 0.5.

**NDCG (при градуальной релевантности):** пусть релевантность: d1=0, d2=2, d3=0, d4=1, d5=0.  
DCG@5 = (2^2-1)/log2(2) + (2^1-1)/log2(4) = 3/1 + 1/2 = 3.5.  
Идеальный порядок: [d2(2), d4(1), d1(0), d3(0), d5(0)] → IDCG@5 = 3/1 + 1/2 = 3.5.  
NDCG@5 = 3.5/3.5 = 1.0 (в этом примере порядок совпал с идеальным, но так бывает не всегда).

---

**Код для расчёта Precision@k и Recall@k:**

```python
def precision_recall_at_k(retrieved, relevant, k):
    """
    retrieved: список документов в порядке ранжирования
    relevant: множество релевантных документов
    k: число рассматриваемых документов
    Возвращает (precision@k, recall@k)
    """
    retrieved_k = retrieved[:k]
    relevant_retrieved = set(retrieved_k) & relevant
    precision = len(relevant_retrieved) / k
    recall = len(relevant_retrieved) / len(relevant) if relevant else 0.0
    return precision, recall

# Пример использования
retrieved = ['d1', 'd2', 'd3', 'd4', 'd5']
relevant = {'d2', 'd4'}
k = 3
p, r = precision_recall_at_k(retrieved, relevant, k)
print(f"Precision@{k}: {p:.3f}, Recall@{k}: {r:.3f}")
```

---

### 3.2. Метрики оценки генерации (Generation Metrics)

Оценка сгенерированного ответа в RAG сложнее, чем оценка поиска, потому что ответ должен быть не только релевантным, но и фактически верным, связным и полезным. В этом разделе мы рассмотрим метрики, которые можно вычислить автоматически, без привлечения человека, а также обсудим их ограничения.

#### 3.2.1. Faithfulness (Верность фактам)

**Определение:** степень, в которой сгенерированный ответ согласуется с фактами, приведёнными в предоставленном контексте (найденных документах). Иными словами, это мера отсутствия галлюцинаций.

**Способы измерения:**
- **На основе NLI (Natural Language Inference).** Каждое утверждение (предложение или клауза) в ответе проверяется на отношение к контексту: следует ли утверждение из контекста (entailment), противоречит ему (contradiction) или нейтрально. Доля утверждений с entailment даёт оценку faithfulness. Используются предобученные NLI-модели (например, `microsoft/deberta-v2-xlarge-mnli`).
- **На основе LLM‑as‑a‑judge.** Мощная LLM (GPT‑4, Claude) получает промпт с контекстом и ответом и выдаёт оценку по шкале (например, от 0 до 1), оценивая, насколько ответ основан на контексте. Этот подход даёт более гибкую оценку, но требует калибровки.

**Интерпретация:** Faithfulness = 0.9 означает, что 90% фактов в ответе подтверждаются контекстом. Низкая faithfulness сигнализирует о галлюцинациях, что часто связано либо с нерелевантным контекстом, либо с неудачным промптом.

#### 3.2.2. Answer Relevance (Релевантность ответа)

**Определение:** насколько сгенерированный ответ прямо отвечает на поставленный вопрос, независимо от его фактической правильности. Оценивается семантическая близость между вопросом и ответом.

**Способы измерения:**
- **Косинусное сходство эмбеддингов.** Вектор вопроса и вектор ответа кодируются с помощью модели эмбеддингов (например, `all‑mpnet‑base‑v2`), затем вычисляется косинусная близость. Это быстрый и интерпретируемый метод.
- **LLM‑as‑a‑judge.** LLM оценивает, насколько ответ соответствует вопросу, учитывая смысл, а не только лексику.

**Интерпретация:** Высокая Answer Relevance означает, что ответ не уходит в сторону и даёт информацию, запрошенную пользователем. Низкая релевантность может указывать на непонимание вопроса LLM или на то, что контекст не содержит нужной информации.

#### 3.2.3. Context Relevance (Релевантность контекста)

**Определение:** насколько найденные ретривером чанки действительно полезны для ответа на вопрос. Это фактически метрика качества поиска, но в RAG она часто вычисляется на уровне генерации, чтобы отделить проблемы ретривера от проблем LLM.

**Способы измерения:**
- **Косинусное сходство между эмбеддингом вопроса и каждого чанка** (среднее или максимальное значение).
- **LLM‑as‑a‑judge** оценивает, содержит ли контекст информацию, необходимую для ответа.

**Интерпретация:** Низкая Context Relevance при высокой Answer Relevance может означать, что LLM смогла ответить из своих внутренних знаний, а не из контекста — это нежелательно, так как снижает проверяемость. Низкая Context Relevance обычно требует улучшения ретривера или чанкинга.

#### 3.2.4. Автоматические метрики с эталоном (BLEU, ROUGE, METEOR)

Эти метрики сравнивают сгенерированный ответ с одним или несколькими эталонными (reference) ответами. Они широко используются в машинном переводе и суммаризации, но имеют серьёзные ограничения для RAG.

- **BLEU (Bilingual Evaluation Understudy)** — основана на совпадении n‑грамм между кандидатом и эталоном. Вычисляется геометрическое среднее точности для n‑грамм (обычно 1–4) с штрафом за длину. BLEU хорошо коррелирует с человеческой оценкой для перевода, но плохо для открытых генеративных задач, так как не учитывает смысл и синонимы.
- **ROUGE (Recall‑Oriented Understudy for Gisting Evaluation)** — семейство метрик, основанных на совпадении n‑грамм (ROUGE‑N), самой длинной общей подпоследовательности (ROUGE‑L) и взвешенной LCS (ROUGE‑W). Лучше подходит для суммаризации, чем BLEU.
- **METEOR** — учитывает синонимы, стемминг и порядок слов, показывая более высокую корреляцию с человеком, чем BLEU.

**Ограничения для RAG:**
1. Требуют эталонного ответа, которого часто нет в реальных задачах.
2. Не оценивают фактическую достоверность — ответ может быть грамматически близок к эталону, но содержать ложные факты.
3. Не учитывают, что один вопрос может иметь множество правильных ответов, и эталон может не покрывать все.

#### 3.2.5. LLM‑as‑a‑judge (Оценка с помощью LLM)

**Определение:** использование мощной LLM (например, GPT‑4, Claude 3) для оценки качества сгенерированного ответа по заданным критериям.

**Как работает:**  
Разработчик создаёт промпт, в котором просит LLM оценить ответ по шкале (например, 1–10) по таким аспектам, как: корректность фактов, полнота, полезность, отсутствие галлюцинаций, стиль. В промпт также включается контекст (найденные документы) и вопрос пользователя. Модель генерирует оценку и, возможно, пояснение.

**Преимущества:**
- Высокая корреляция с человеческой оценкой (часто выше, чем у BLEU/ROUGE).
- Возможность оценивать сложные, многомерные аспекты.
- Не требует эталонных ответов.

**Недостатки:**
- Стоимость: использование больших моделей через API дорого.
- Зависимость от выбора модели и формулировки промпта; разные модели могут давать разные оценки.
- Предвзятость: модель может быть снисходительна или строга, что требует калибровки.
- Отсутствие прозрачности: сложно понять, почему модель поставила именно такую оценку (хотя можно попросить пояснение).

#### 3.2.6. Человеческая оценка (Human Evaluation)

Золотой стандарт, особенно для финального тестирования перед релизом. Эксперты или краудворкеры оценивают ответы по шкале (например, Likert 1–5) по различным критериям (корректность, полнота, стиль). Человеческая оценка дорога и медленна, поэтому её используют для валидации автоматических метрик и для сравнения критических версий системы.

---

### 3.3. Комплексные фреймворки оценки

Современные RAG-системы редко оценивают вручную — для этого существуют специализированные фреймворки, которые автоматизируют вычисление метрик, генерацию тестовых данных и мониторинг. Рассмотрим четыре наиболее влиятельных подхода с детальным теоретическим разбором каждого и практическими примерами.

#### 3.3.1. RAGAS (Retrieval-Augmented Generation Assessment)

**Теоретическая основа.** RAGAS (Es et al., 2023) — это открытый фреймворк, предназначенный для автоматической оценки RAG-систем **без эталонных ответов**. Он вычисляет три ключевые метрики, используя LLM в качестве судьи:

- **Faithfulness** — проверяется каждое утверждение в ответе на entialment относительно контекста с помощью NLI-модели или LLM.
- **Answer Relevance** — вычисляется как косинусное сходство между эмбеддингами вопроса и ответа (после нормализации).
- **Context Relevance** — вычисляется как средняя косинусная близость между эмбеддингом вопроса и эмбеддингами всех чанков (или доля чанков с близостью выше порога).

RAGAS не требует эталонных ответов, но требует наличия контекста (результатов ретривера) и сгенерированного ответа для каждого вопроса. Это делает его удобным для исследовательских экспериментов и быстрой итерации.

**Практический пример запуска RAGAS:**

```python
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_relevancy
from datasets import Dataset

# Подготовка данных: список словарей с полями "question", "answer", "contexts"
data = {
    "question": ["Какие налоги платят самозанятые?"],
    "answer": ["Самозанятые платят налог на профессиональный доход по ставке 4% или 6%."],
    "contexts": [["НПД — специальный налоговый режим для самозанятых. Ставка 4% при работе с физлицами, 6% — с юрлицами."]]
}
dataset = Dataset.from_dict(data)

# Вычисление метрик
result = evaluate(dataset, metrics=[faithfulness, answer_relevancy, context_relevancy])
print(result)
```

**Интерпретация результатов:** Значения от 0 до 1, где 1 — идеально. Если faithfulness низкая, значит, LLM галлюцинирует. Если answer_relevancy низкая, ответ не отвечает на вопрос. Если context_relevancy низкая, ретривер нашёл нерелевантные чанки.

---

#### 3.3.2. ARES (Automatic RAG Evaluation System)

**Теоретическая основа.** ARES (Es et al., 2023) — это более сложный фреймворк, который решает проблему **отсутствия размеченных данных** путём генерации синтетического датасета на основе самого корпуса документов.

**Внутренний механизм:**
1. **Генерация синтетических данных.** Используя мощную LLM (например, GPT‑4), для каждого документа (или чанка) генерируется вопрос, на который этот документ является идеальным ответом, и эталонный ответ, составленный из содержимого документа. Таким образом создаётся большой набор пар «вопрос – контекст – эталонный ответ», который служит прокси для реальных пользовательских запросов.

2. **LLM‑as‑a‑judge с верификацией.** ARES использует LLM-судью для оценки ответов тестируемой системы на синтетических вопросах. Однако, чтобы компенсировать возможные ошибки судьи, ARES вводит этап **калибровки**: на небольшом подмножестве вручную размеченных примеров (обычно 200–300) оцениваются точность и смещение судьи. Эти оценки используются для коррекции результатов на всём синтетическом наборе.

3. **Статистическая оценка.** ARES выдаёт не точечную оценку, а **доверительные интервалы** для метрик (например, для Faithfulness). Это достигается с помощью бутстрапа (bootstrap) по синтетическому датасету, что позволяет разработчику с заданной вероятностью утверждать, что реальное качество лежит в определённом диапазоне.

**Ключевое преимущество:** ARES значительно сокращает затраты на ручную разметку, сохраняя при этом статистическую обоснованность оценок. Он особенно полезен для корпоративных систем, где данные часто меняются и нет возможности каждый раз размечать новые примеры.

---

#### 3.3.3. TruLens

**Теоретическая основа.** TruLens — это фреймворк, ориентированный на **непрерывный мониторинг и глубокую диагностику** RAG-приложений в продакшене. Его архитектура основана на трассировке выполнения цепочек вызовов.

**Внутренний механизм:**
1. **Трассировка (Instrumentation).** TruLens автоматически обёртывает ретривер и LLM, записывая каждый шаг: входной запрос, извлечённые чанки (с метаданными и скорами), сгенерированный ответ, время выполнения и потребление токенов. Эта информация сохраняется в хранилище (локальном или удалённом), формируя историю работы системы.

2. **Функции обратной связи (Feedback Functions).** TruLens реализует метрики как функции, которые могут выполняться как синхронно (во время ответа пользователю), так и асинхронно (в фоновом режиме). Эти функции используют как классические эмбеддинги, так и LLM-промпты, и возвращают числовую оценку или комментарий. Благодаря трассировке каждая метрика может быть привязана к конкретному этапу пайплайна.

3. **Каузальный анализ.** Благодаря сохранённым трассировкам, разработчик может для любого запроса увидеть полную цепочку: какой был запрос, какие чанки нашлись, какой ответ сгенерировала LLM, и какую оценку поставила каждая метрика. Это позволяет проводить корневой анализ ошибок (Root Cause Analysis) на уровне отдельных сущностей, а не только агрегированных чисел.

**Сравнение версий.** TruLens позволяет прогонять набор «золотых» вопросов через разные версии системы (например, после изменения ретривера) и визуально сравнивать метрики, что облегчает регрессионное тестирование.

---

#### 3.3.4. DeepEval

**Теоретическая основа.** DeepEval — это фреймворк, интегрирующий оценку качества непосредственно в процесс разработки, вдохновлённый Test-Driven Development (TDD). Он позиционируется как библиотека для **модульного тестирования** LLM-приложений.

**Внутренний механизм:**
1. **Абстракция Test Case.** DeepEval вводит понятие тест-кейса, который связывает входные данные с ожидаемыми метриками. Тест-кейс может содержать эталонный ответ (для BLEU/ROUGE), или просто вопрос и контекст (для Faithfulness/Relevancy). DeepEval использует собственный движок вычисления метрик, что делает его независимым от внешних библиотек.

2. **G‑Eval (LLM-based Evaluation).** DeepEval часто использует технику G‑Eval, где метрика вычисляется путём промптинга LLM с цепочкой мыслей (Chain‑of‑Thought). В отличие от простого запроса оценки, G‑Eval просит LLM обосновать оценку, что повышает надёжность (корреляция с человеком выше на 15–20%). Теоретически G‑Eval использует вероятностное моделирование: LLM генерирует оценку, которая затем нормализуется через сигмоиду или softmax над логитами.

3. **Интеграция с CI/CD.** DeepEval предоставляет декораторы, позволяющие превратить любую функцию в тест. Если вычисленная метрика падает ниже заданного порога (например, Faithfulness < 0.8), тест падает, блокируя слияние кода. Это превращает оценку качества из исследовательской задачи в инженерный процесс.

4. **Синтетическая генерация для стресс-тестирования.** DeepEval умеет генерировать синтетические вопросы не просто для оценки, а для тестирования граничных случаев (adversarial testing): вопросы с противоречивой информацией, вопросы, на которые в контексте нет ответа, и т.д. Это позволяет проверить, умеет ли система корректно обрабатывать сложные ситуации.

---

**Сравнительная таблица фреймворков**

| Критерий | RAGAS | ARES | TruLens | DeepEval |
| :--- | :--- | :--- | :--- | :--- |
| **Основная парадигма** | Оценка по готовым данным | Автоматическая генерация эталонов | Непрерывный мониторинг + трассировка | Модульное тестирование (Unit‑testing) |
| **Требования к данным** | Нужны ответы системы и контекст | Нужен только корпус документов | Не требует (работает в рантайме) | Может использовать эталоны или генерировать |
| **Механизм вычислений** | LLM‑as‑a‑judge + эмбеддинги | LLM‑судья с калибровкой и бутстрапом | Асинхронные колбэки на основе трассировки | G‑Eval или локальные вычисления |
| **Учёт времени (Latency)** | Офлайн | Офлайн | Может быть онлайн (влияет) | Офлайн (в тестовой среде) |
| **Глубина диагностики** | Метрики на уровне набора данных | Доверительные интервалы для метрик | Пошаговая визуализация каждого запроса | Проверка порогов на уровне кода (assert) |
| **Целевая аудитория** | Исследователи, Data Scientists | Инженеры с ограниченной разметкой | MLOps, Инженеры по мониторингу | Разработчики (Software Engineers) |

---

### 3.4. Мониторинг качества в продакшене

Оценка в продакшене отличается от офлайн-оценки: данные поступают в реальном времени, и качество может меняться со временем (дрейф данных, устаревание документов). Поэтому необходим непрерывный мониторинг.

#### 3.4.1. Сбор метрик в реальном времени

- **Latency (задержка):** время от запроса до ответа (мс). Важно отслеживать перцентили (p50, p95, p99), так как среднее значение может маскировать редкие, но долгие запросы.
- **QPS (Queries Per Second):** нагрузка на систему. Резкий рост или падение может сигнализировать о проблемах.
- **Доля успешных ответов:** процент запросов, на которые система вернула ответ без ошибок (таймауты, исключения).
- **Доля отказов (no documents found):** процент запросов, для которых ретривер не нашёл ни одного чанка выше порога релевантности. Если доля растёт, это может указывать на устаревание индекса или изменение характера запросов.
- **Доля негативной обратной связи:** если пользователи могут ставить лайки/дизлайки, эта метрика — прямой индикатор удовлетворённости.

#### 3.4.2. Анализ ошибок

Ошибки можно классифицировать на три типа:
- **Поисковые (retrieval errors):** релевантные документы существуют в корпусе, но не были найдены (низкий Recall). Причины: неподходящая модель эмбеддингов, слишком маленький `k`, неправильный чанкинг, отсутствие гибридного поиска.
- **Генерационные (generation errors):** документы найдены, но LLM выдала неверный или неполный ответ. Причины: неудачный промпт, недостаточное контекстное окно, слабая модель.
- **Смешанные:** документы частично релевантны, LLM неправильно интерпретировала их. Требуют комплексного решения.

Для каждого типа разрабатываются конкретные улучшения: добавление реранкинга, увеличение `k`, настройка промпта, смена модели.

#### 3.4.3. А/Б‑тестирование

Перед внедрением изменений (новая модель эмбеддингов, новый реранкер, новый промпт) обязательно проводите А/Б‑тесты. Разбейте трафик на две группы (50%/50%) и сравнивайте ключевые метрики (лайки, время на странице, количество повторных запросов) в течение достаточно длительного периода (минимум неделя), чтобы учесть дневные и недельные колебания.

#### 3.4.4. Дашборды

Визуализируйте ключевые метрики в реальном времени с помощью инструментов, таких как Grafana, Kibana или Metabase. Рекомендуемый набор графиков:
- График latency (p50, p95, p99) за последние 24 часа.
- График QPS.
- Доля отказов (no documents found).
- Распределение оценок пользователей (лайки/дизлайки).
- Топ запросов с низкой оценкой или с отказами — для ручного анализа и приоритизации улучшений.

---

### 3.5. Контрольные вопросы

1. *В чём разница между Precision@k и Recall@k, и почему они часто используются вместе?*  
   **Ответ:** Precision@k измеряет долю релевантных документов среди возвращённых, а Recall@k — долю найденных релевантных от всех релевантных. Вместе они дают полную картину: высокая Precision при низком Recall означает, что мы находим мало, но точно; высокий Recall при низкой Precision — находим много, но с шумом. Использование обеих метрик позволяет сбалансировать качество.

2. *Почему NDCG лучше, чем Precision@k, для задач ранжирования?*  
   **Ответ:** NDCG учитывает градуальную релевантность (не только бинарную), а также позицию документа с логарифмическим дисконтированием. Это более тонко отражает качество ранжирования, особенно когда важна не только топ‑1, но и порядок остальных результатов. Precision@k игнорирует порядок внутри k и не различает документы по степени релевантности.

3. *Какие ограничения у LLM‑as‑a‑judge и как их можно смягчить?*  
   **Ответ:** Ограничения: высокая стоимость, зависимость от выбора модели, возможная предвзятость, чувствительность к формулировке промпта. Смягчение: использовать несколько моделей и усреднять оценки, тщательно калибровать промпты, периодически валидировать оценки на человеческих данных (как в ARES), применять техники G‑Eval для повышения надёжности.

---

### 3.6. Задания (для самостоятельной работы)

1. **Ручной расчёт метрик.** Дан корпус из 6 документов `[d1..d6]`. Для запроса релевантны `{d1, d3, d5}`. Ретривер вернул: `[d2, d1, d4, d3, d6, d5]`. Рассчитайте Precision@3, Recall@3, Precision@5, Recall@5, MRR, MAP. *(Ответы: Precision@3=1/3, Recall@3=1/3, Precision@5=2/5=0.4, Recall@5=2/3≈0.667, MRR=1/2=0.5, MAP=(1/2 + 2/4)/3 = (0.5+0.5)/3≈0.333)*

2. **Практическое использование RAGAS.** Установите RAGAS (`pip install ragas`) и на своём небольшом датасете (10–20 вопросов с контекстами и ответами) запустите оценку по метрикам faithfulness, answer_relevancy, context_relevancy. Проанализируйте полученные значения и напишите рекомендации по улучшению системы, если какая-то метрика ниже 0.8.

---

### 3.7. Список литературы

1. **Es, S., et al. (2023).** *RAGAS: Automated Evaluation of Retrieval-Augmented Generation.* – arXiv:2309.15217.
2. **Es, S., et al. (2023).** *ARES: Automatic RAG Evaluation with Synthetic Data.* – (дополнительная работа, описывающая синтетическую генерацию).
3. **TruLens Documentation.** – https://www.trulens.org/ – теоретические основы трассировки и обратной связи.
4. **DeepEval Documentation.** – https://docs.confident-ai.com/ – описание G‑Eval и модульного тестирования.
5. **Liu, N. et al. (2024).** *Evaluating RAG Systems: A Comprehensive Survey.* – arXiv:2405.12345.
6. **Hugging Face Blog.** *Evaluating RAG with RAGAS.* – https://huggingface.co/blog/rag-evaluation – обзор метрик.
7. **Wang, A., et al. (2020).** *GLUE: A Multi-Task Benchmark and Analysis Platform for Natural Language Understanding.* – статья, описывающая NLI-модели, используемые для Faithfulness.



## Тема 4. Предобработка данных для RAG (Ingestion)

Качество RAG-системы напрямую зависит от качества данных, которые в неё загружаются. Плохо извлечённый, неочищенный или неправильно структурированный текст приведёт к низкому качеству поиска и, как следствие, к неверным или неполным ответам. В этой теме мы детально разберём весь ETL-пайплайн (Extract, Transform, Load) для RAG: от извлечения текста из различных форматов до сохранения очищенных данных с метаданными, готовых к чанкингу и векторизации. Материал ориентирован на продакшен-реализацию с готовыми решениями и примерами кода.

---

### 4.1. Извлечение текста из разных форматов

Первый этап — извлечение текстового содержимого из исходных файлов. Корпоративные данные редко хранятся в виде чистых `.txt` файлов: это могут быть PDF-отчёты, Word-документы, HTML-страницы, Excel-таблицы, Markdown-файлы и другие форматы. Выбор правильного инструмента для каждого формата критически важен, так как от этого зависит полнота и точность извлечения.

#### 4.1.1. Работа с PDF-файлами

PDF — один из самых сложных форматов для извлечения текста, так как он хранит информацию о позиционировании текста на странице, а не логическую структуру. Существует несколько библиотек, каждая со своими сильными и слабыми сторонами.

| Библиотека | Способ извлечения | Точность | Скорость | Работа с таблицами | Поддержка сканов |
|------------|-------------------|----------|----------|-------------------|------------------|
| **PyPDF2 / pypdf** | Извлечение по потокам | Средняя | Высокая | Плохая | Нет (только текст) |
| **pdfplumber** | Анализ позиционирования букв | Высокая | Средняя | Отличная | Нет (только текст) |
| **PyMuPDF (fitz)** | Извлечение по потокам + анализ | Высокая | Высокая | Хорошая | Нет (только текст) |
| **pypdf + OCR (Tesseract)** | OCR для сканов | Зависит от OCR | Низкая | Плохая | Да (с OCR) |

**Рекомендации:**
- Для обычных текстовых PDF (созданных из Word, LaTeX) используйте **pypdf** (наследник PyPDF2) или **pdfplumber**, если нужны таблицы.
- Для PDF с таблицами используйте **pdfplumber** — он умеет определять границы ячеек и извлекать таблицы в структурированном виде.
- Для сканированных PDF (книги, архивные документы) используйте **pypdf + OCR** (например, Tesseract) или специализированные сервисы (AWS Textract, Google Document AI).

**Пример кода для извлечения текста из PDF:**

```python
import pdfplumber
from pypdf import PdfReader
import logging

logger = logging.getLogger(__name__)

def extract_pdf_with_fallback(file_path: str) -> str:
    """
    Извлекает текст из PDF, используя pdfplumber, с fallback на pypdf при ошибке.
    """
    text = ""
    try:
        # Попытка извлечения через pdfplumber (лучше для сложной вёрстки)
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
        if text.strip():
            logger.info(f"PDF извлечён через pdfplumber: {file_path}")
            return text
    except Exception as e:
        logger.warning(f"pdfplumber не сработал для {file_path}: {e}")

    try:
        # Fallback на pypdf (быстрее, но может потерять структуру)
        with open(file_path, "rb") as f:
            reader = PdfReader(f)
            for page in reader.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
        logger.info(f"PDF извлечён через pypdf (fallback): {file_path}")
    except Exception as e:
        logger.error(f"Не удалось извлечь текст из PDF {file_path}: {e}")
        text = ""

    return text.strip()
```

**Извлечение таблиц из PDF с помощью pdfplumber:**

```python
def extract_pdf_tables(file_path: str) -> list:
    """Извлекает все таблицы из PDF в виде списка pandas DataFrame."""
    import pandas as pd
    tables = []
    with pdfplumber.open(file_path) as pdf:
        for i, page in enumerate(pdf.pages):
            page_tables = page.extract_tables()
            for j, table in enumerate(page_tables):
                if table and len(table) > 1:  # Пропускаем пустые и однострочные
                    df = pd.DataFrame(table[1:], columns=table[0])
                    tables.append(df)
                    logger.debug(f"Страница {i+1}, таблица {j+1}: {df.shape}")
    return tables
```

#### 4.1.2. Работа с .docx (Microsoft Word)

Библиотека `python-docx` позволяет извлекать текст с сохранением структуры абзацев, таблиц и стилей.

**Пример кода:**

```python
from docx import Document

def extract_docx(file_path: str) -> str:
    """Извлекает текст из .docx с сохранением структуры абзацев."""
    try:
        doc = Document(file_path)
        paragraphs = []
        for para in doc.paragraphs:
            if para.text.strip():
                paragraphs.append(para.text)
        # Добавляем текст из таблиц
        for table in doc.tables:
            for row in table.rows:
                row_text = " | ".join(cell.text.strip() for cell in row.cells if cell.text.strip())
                if row_text:
                    paragraphs.append(row_text)
        return "\n\n".join(paragraphs)
    except Exception as e:
        logger.error(f"Ошибка при извлечении .docx {file_path}: {e}")
        return ""
```

#### 4.1.3. Работа с HTML

Для извлечения основного текста из HTML-страниц (без рекламы, навигации, скриптов) лучше всего использовать `beautifulsoup4` с `readability-lxml` для очистки.

**Пример кода:**

```python
from bs4 import BeautifulSoup
import requests
from readability import Document  # pip install readability-lxml

def extract_html(html_content: str) -> str:
    """Извлекает основной текст из HTML (с очисткой от шума)."""
    try:
        doc = Document(html_content)
        return doc.summary()  # Возвращает HTML очищенного текста
    except Exception as e:
        # Fallback: просто извлекаем весь текст
        soup = BeautifulSoup(html_content, "html.parser")
        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()
        return soup.get_text(separator="\n")
```

#### 4.1.4. Работа с Markdown

Markdown легко парсится, так как это текстовый формат. Можно просто читать файл, а можно использовать парсер для извлечения структуры.

**Пример кода:**

```python
import markdown  # pip install markdown

def extract_markdown(file_path: str) -> str:
    """Извлекает текст из Markdown, конвертируя в HTML, а затем в текст."""
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            md_text = f.read()
        # Конвертируем Markdown в HTML
        html = markdown.markdown(md_text)
        # Извлекаем текст из HTML
        soup = BeautifulSoup(html, "html.parser")
        return soup.get_text(separator="\n")
    except Exception as e:
        logger.error(f"Ошибка при извлечении Markdown {file_path}: {e}")
        return ""
```

#### 4.1.5. Работа с Excel и CSV

Для табличных данных используем `pandas`. Важно извлекать не только значения, но и заголовки столбцов, чтобы сохранить смысл.

**Пример кода:**

```python
import pandas as pd

def extract_excel(file_path: str) -> str:
    """Извлекает текст из Excel-файла, формируя текстовое представление таблиц."""
    try:
        df = pd.read_excel(file_path, sheet_name=None)  # читаем все листы
        text_parts = []
        for sheet_name, sheet_df in df.items():
            text_parts.append(f"[Лист: {sheet_name}]")
            # Преобразуем DataFrame в текст с заголовками
            for _, row in sheet_df.iterrows():
                row_text = " | ".join(str(val) for val in row.values if pd.notna(val))
                if row_text:
                    text_parts.append(row_text)
        return "\n".join(text_parts)
    except Exception as e:
        logger.error(f"Ошибка при извлечении Excel {file_path}: {e}")
        return ""
```

---

### 4.2. Очистка и нормализация текста

Извлечённый текст часто содержит шум: лишние пробелы, спецсимволы, невидимые символы, ошибки кодировки. Очистка — критический этап, от которого зависит качество эмбеддингов и поиска.

#### 4.2.1. Удаление спецсимволов и лишних пробелов

```python
import re

def clean_text(text: str) -> str:
    """
    Базовая очистка текста:
    - Удаление невидимых символов
    - Нормализация пробелов
    - Удаление управляющих символов
    """
    if not text:
        return ""

    # Удаляем невидимые символы (ASCII 0-31, 127)
    text = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]', '', text)

    # Заменяем все виды пробелов на обычный
    text = re.sub(r'\s+', ' ', text).strip()

    # Удаляем лишние символы в начале/конце
    text = text.strip()

    return text
```

#### 4.2.2. Приведение к нижнему регистру

Для лексического поиска (BM25, TF‑IDF) часто приводят текст к нижнему регистру, чтобы избежать чувствительности к регистру. Однако для векторного поиска это не обязательно, так как модели эмбеддингов обычно учитывают регистр в меньшей степени.

```python
def normalize_case(text: str, for_lexical_search: bool = False) -> str:
    """Приводит текст к нижнему регистру, если это необходимо."""
    if for_lexical_search:
        return text.lower()
    return text
```

#### 4.2.3. Удаление стоп-слов

Стоп-слова (предлоги, союзы, местоимения) часто удаляют при лексическом поиске, чтобы уменьшить шум и ускорить поиск. Однако при векторном поиске они сохраняются, так как модели эмбеддингов учитывают их контекст.

```python
from nltk.corpus import stopwords
import nltk

nltk.download('stopwords')
STOPWORDS_EN = set(stopwords.words('english'))
STOPWORDS_RU = set()  # можно загрузить русские стоп-слова отдельно

def remove_stopwords(text: str, language: str = 'en') -> str:
    """Удаляет стоп-слова из текста (только для лексического поиска)."""
    stopwords_set = STOPWORDS_RU if language == 'ru' else STOPWORDS_EN
    words = text.split()
    filtered_words = [w for w in words if w.lower() not in stopwords_set]
    return " ".join(filtered_words)
```

#### 4.2.4. Лемматизация и стемминг

- **Стемминг** — упрощённое усечение слов до корня (например, "running" → "run", "beautiful" → "beauti"). Быстрый, но грубый метод.
- **Лемматизация** — приведение слова к нормальной форме (например, "running" → "run", "better" → "good"). Более точный, но медленный метод, требует словаря.

**Для английского языка** используем `spaCy`:

```python
import spacy

nlp_en = spacy.load("en_core_web_sm")

def lemmatize_en(text: str) -> str:
    """Лемматизация текста на английском."""
    doc = nlp_en(text)
    return " ".join(token.lemma_ for token in doc if not token.is_punct)
```

**Для русского языка** используем `pymorphy2`:

```python
import pymorphy2

morph = pymorphy2.MorphAnalyzer()

def lemmatize_ru(text: str) -> str:
    """Лемматизация текста на русском."""
    words = text.split()
    lemmas = []
    for word in words:
        parsed = morph.parse(word)[0]
        lemmas.append(parsed.normal_form)
    return " ".join(lemmas)
```

**Сравнение стемминга и лемматизации:**

| Критерий | Стемминг | Лемматизация |
|----------|----------|--------------|
| Скорость | Очень высокая | Медленнее (в 2-3 раза) |
| Точность | Грубая, иногда ошибки | Высокая |
| Ресурсы | Минимальные | Нужен словарь/модель |
| Когда использовать | Для лексического поиска (BM25) | Для точного анализа, поиска по смыслу |

#### 4.2.5. Обработка таблиц и списков

Таблицы и списки содержат структурированную информацию, которую важно сохранить при преобразовании в текст.

```python
def format_table_as_text(table: list) -> str:
    """Преобразует таблицу (список строк) в текстовое представление."""
    if not table:
        return ""
    # Определяем ширину столбцов
    col_widths = [max(len(str(row[i])) for row in table) for i in range(len(table[0]))]
    lines = []
    for row in table:
        line = " | ".join(str(cell).ljust(col_widths[i]) for i, cell in enumerate(row))
        lines.append(line)
    return "\n".join(lines)

def format_list_as_text(items: list) -> str:
    """Преобразует список в текст с маркерами."""
    return "\n".join(f"• {item}" for item in items)
```

#### 4.2.6. Полная функция очистки

```python
def full_clean_pipeline(text: str, language: str = 'en') -> str:
    """
    Полный пайплайн очистки текста.
    """
    if not text:
        return ""

    # Базовые шаги (всегда)
    text = clean_text(text)

    # Для лексического поиска (опционально)
    # text = normalize_case(text, for_lexical_search=True)
    # text = remove_stopwords(text, language)

    # Для лемматизации (опционально, если нужна нормализация)
    if language == 'ru':
        text = lemmatize_ru(text)
    else:
        text = lemmatize_en(text)

    return text
```

---

### 4.3. Работа с метаданными документов

Метаданные — это структурированная информация о документе, которая не является частью основного текста, но критически важна для фильтрации, цитирования и управления версиями. Сохранение метаданных позволяет пользователю задавать уточняющие вопросы ("покажи только из документов за 2024 год") и проверять источники.

**Рекомендуемый набор метаданных:**

| Поле | Тип | Описание | Пример |
|------|-----|----------|--------|
| `source` | str | Имя файла или URL | `"report_2024.pdf"` |
| `file_path` | str | Полный путь к файлу | `"/data/reports/report_2024.pdf"` |
| `doc_id` | str | Уникальный идентификатор | `"doc_001"` |
| `title` | str | Заголовок документа | `"Годовой отчёт 2024"` |
| `author` | str | Автор | `"Иванов И.И."` |
| `created_date` | str | Дата создания | `"2024-01-15"` |
| `modified_date` | str | Дата изменения | `"2024-12-10"` |
| `language` | str | Язык документа | `"ru"` |
| `version` | str | Версия документа | `"v2.3"` |
| `page_number` | int | Номер страницы (для чанка) | `12` |
| `section` | str | Раздел/глава | `"Глава 3. Налогообложение"` |
| `tags` | list | Теги/категории | `["финансы", "налог"]` |
| `department` | str | Отдел | `"Бухгалтерия"` |
| `is_active` | bool | Актуален ли документ | `True` |

**Структура для хранения в векторной БД (Chroma):**

```python
from dataclasses import dataclass
from typing import List, Optional
from datetime import datetime

@dataclass
class DocumentMetadata:
    source: str
    file_path: str
    doc_id: str
    title: str = ""
    author: str = ""
    created_date: Optional[str] = None
    modified_date: Optional[str] = None
    language: str = "en"
    version: str = "1.0"
    page_number: int = 0
    section: str = ""
    tags: List[str] = None
    department: str = ""
    is_active: bool = True

    def to_dict(self) -> dict:
        """Преобразует в словарь для сохранения в БД."""
        return {
            "source": self.source,
            "file_path": self.file_path,
            "doc_id": self.doc_id,
            "title": self.title,
            "author": self.author,
            "created_date": self.created_date,
            "modified_date": self.modified_date,
            "language": self.language,
            "version": self.version,
            "page_number": self.page_number,
            "section": self.section,
            "tags": self.tags or [],
            "department": self.department,
            "is_active": str(self.is_active).lower(),
        }
```

**Пример фильтрации по метаданным в Chroma:**

```python
import chromadb

client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_collection("documents")

# Поиск только в документах за 2024 год
results = collection.query(
    query_texts=["налог на прибыль"],
    n_results=10,
    where={
        "$and": [
            {"created_date": {"$gte": "2024-01-01"}},
            {"created_date": {"$lte": "2024-12-31"}}
        ]
    }
)

# Фильтрация по автору
results = collection.query(
    query_texts=["налог на прибыль"],
    n_results=10,
    where={"author": "Иванов И.И."}
)
```

---

### 4.4. Работа с мультиязычными документами

Корпоративные данные могут быть на разных языках. Для RAG это создаёт дополнительную сложность: нужно правильно определять язык, использовать подходящие модели эмбеддингов и, возможно, выполнять перевод.

#### 4.4.1. Определение языка

```python
from langdetect import detect, DetectorFactory
import fasttext

# Для langdetect
DetectorFactory.seed = 42

def detect_language_langdetect(text: str) -> str:
    """Определяет язык текста с помощью langdetect."""
    try:
        return detect(text)
    except:
        return "unknown"

# Для fasttext (требует скачивания модели)
# model = fasttext.load_model('lid.176.bin')
def detect_language_fasttext(text: str) -> str:
    """Определяет язык текста с помощью fasttext."""
    # prediction = model.predict(text)
    # return prediction[0][0].replace('__label__', '')
    pass
```

**Сравнение методов:**

| Библиотека | Точность | Скорость | Языки | Требования |
|------------|----------|----------|-------|------------|
| **langdetect** | Высокая (для основных языков) | Средняя | 55+ | Нет (чистый Python) |
| **fasttext** | Очень высокая | Высокая | 176 | Нужно скачать модель (100+ МБ) |
| **spaCy** | Высокая | Низкая | ~20 | Нужна языковая модель |

**Рекомендация:** для быстрой проверки используйте `langdetect`, для продакшена — `fasttext`.

#### 4.4.2. Мультиязычные модели эмбеддингов

| Модель | Размерность | Языки | MTEB (среднее) | Примечание |
|--------|-------------|-------|----------------|------------|
| **intfloat/multilingual-e5-large** | 1024 | 100+ | ~65 | Лучшая open‑source, требует нормализации |
| **intfloat/multilingual-e5-base** | 768 | 100+ | ~63 | Баланс скорость/качество |
| **LaBSE** | 768 | 109 | ~58 | Хороша для перевода, но устаревает |
| **paraphrase-multilingual-MiniLM-L12-v2** | 384 | 50+ | ~57 | Быстрая, подходит для прототипов |
| **BAAI/bge-m3** | 1024 | 100+ | ~66 | Новая, отличное качество |

**Рекомендация:** для продакшена используйте `intfloat/multilingual-e5-large` или `BAAI/bge-m3`.

**Пример генерации эмбеддингов для мультиязычных документов:**

```python
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/multilingual-e5-large")

def get_embedding(text: str) -> list:
    """Генерирует эмбеддинг для текста на любом языке."""
    # Для E5 моделей требуется префикс "query: " или "passage: "
    embedding = model.encode("passage: " + text, normalize_embeddings=True)
    return embedding.tolist()
```

#### 4.4.3. Особенности русского языка

Русская морфология (падежи, склонения, спряжения) делает лексический поиск сложным — одно и то же слово может иметь множество форм. Поэтому для русского языка:

- Обязательно используйте **лемматизацию** (pymorphy2) или стемминг перед лексическим поиском.
- Для векторного поиска лучше использовать мультиязычные модели, которые обучались на русских текстах (например, multilingual-e5).
- Учитывайте, что модели эмбеддингов для русского языка могут иметь меньшую точность, чем для английского — это нормально, компенсируется качественным чанкингом и реранкингом.

---

### 4.5. Real‑time индексация и управление версиями

В продакшене документы не статичны: они добавляются, обновляются и удаляются. Система должна поддерживать инкрементальное обновление индекса без остановки работы.

#### 4.5.1. Инкрементальное обновление

**Проблема:** при добавлении новых документов не хочется перестраивать весь индекс заново.

**Решение (Chroma):** просто добавляем новые документы с новыми ID.

```python
collection.add(
    documents=[new_text],
    metadatas=[new_metadata],
    ids=[new_doc_id],
    embeddings=[new_embedding]  # опционально, можно сгенерировать внутри
)
```

**Решение (FAISS):** используем `IndexIDMap` для поддержки обновлений.

```python
import faiss
import numpy as np

# Создаём индекс
index = faiss.IndexFlatIP(dimension)  # или HNSW
id_map = faiss.IndexIDMap(index)

# Добавляем векторы с ID
embeddings = np.array([new_embedding]).astype('float32')
id_map.add_with_ids(embeddings, np.array([new_doc_id]))
```

#### 4.5.2. Удаление устаревших чанков

**Стратегия мягкого удаления:** не удаляем физически, а помечаем `is_active=False` в метаданных и фильтруем при поиске.

```python
# При поиске добавляем фильтр
results = collection.query(
    query_texts=["запрос"],
    n_results=10,
    where={"is_active": "true"}  # только активные
)
```

**Для FAISS** физическое удаление сложнее, поэтому мягкое удаление — основной подход.

#### 4.5.3. Версионирование

Храните версию документа в метаданных и при поиске выдавайте только последние версии.

```python
# При поиске используем фильтр по версии
results = collection.query(
    query_texts=["запрос"],
    n_results=10,
    where={"version": "latest"}
)
```

---

### 4.6. ETL-пайплайны (кратко)

Для регулярной индексации больших объёмов данных используют ETL-фреймворки:

- **Apache Airflow** — самый популярный, позволяет строить DAG (Directed Acyclic Graph) задач.
- **Prefect** — более современный, с лучшим UX и поддержкой асинхронности.
- **Dagster** — фокусируется на качественном тестировании и валидации данных.

**Пример простого DAG для Airflow:**

```python
from airflow import DAG
from airflow.operators.python_operator import PythonOperator
from datetime import datetime

default_args = {'owner': 'data_team', 'start_date': datetime(2024, 1, 1)}

dag = DAG('rag_ingestion', default_args=default_args, schedule_interval='@daily')

def extract_pdfs():
    # Загрузка PDF из папки
    pass

def clean_and_chunk():
    # Очистка и чанкинг
    pass

def embed_and_index():
    # Генерация эмбеддингов и индексация
    pass

extract = PythonOperator(task_id='extract', python_callable=extract_pdfs, dag=dag)
clean = PythonOperator(task_id='clean', python_callable=clean_and_chunk, dag=dag)
index = PythonOperator(task_id='index', python_callable=embed_and_index, dag=dag)

extract >> clean >> index
```

---

### 4.7. Практический пример: полный скрипт индексации

Ниже представлен полный скрипт, который загружает все PDF из папки, извлекает текст и метаданные, очищает их и сохраняет в JSON.

```python

!pip install pypdf pdfplumber

import os
import json
import logging
from datetime import datetime
from typing import Dict, List
from pathlib import Path

import pdfplumber
from pypdf import PdfReader
import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def extract_pdf_text(file_path: str) -> str:
    """Извлекает текст из PDF с fallback."""
    text = ""
    try:
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
        if text.strip():
            return text
    except Exception as e:
        logger.warning(f"pdfplumber не сработал для {file_path}: {e}")

    try:
        with open(file_path, "rb") as f:
            reader = PdfReader(f)
            for page in reader.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
    except Exception as e:
        logger.error(f"Не удалось извлечь текст из {file_path}: {e}")
        return ""

    return text.strip()

def clean_text(text: str) -> str:
    """Очищает текст от шума."""
    import re
    if not text:
        return ""
    # Удаляем управляющие символы
    text = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]', '', text)
    # Нормализуем пробелы
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def get_pdf_metadata(file_path: str) -> Dict:
    """Извлекает метаданные из PDF и файловой системы."""
    stat = os.stat(file_path)
    metadata = {
        "source": os.path.basename(file_path),
        "file_path": str(file_path),
        "doc_id": Path(file_path).stem,
        "created_date": datetime.fromtimestamp(stat.st_ctime).isoformat(),
        "modified_date": datetime.fromtimestamp(stat.st_mtime).isoformat(),
        "page_number": 0,
        "section": "",
        "tags": [],
        "department": "",
        "is_active": True
    }
    try:
        with open(file_path, "rb") as f:
            reader = PdfReader(f)
            info = reader.metadata
            if info:
                metadata["author"] = str(info.get('/Author', ''))
                metadata["title"] = str(info.get('/Title', ''))
    except Exception as e:
        logger.warning(f"Не удалось извлечь метаданные PDF для {file_path}: {e}")
    return metadata

def process_pdf_directory(input_dir: str, output_json: str) -> List[Dict]:
    """Обрабатывает все PDF в директории и сохраняет результат в JSON."""
    results = []
    pdf_files = list(Path(input_dir).glob("**/*.pdf"))

    if not pdf_files:
        logger.warning(f"PDF файлы не найдены в {input_dir}")
        return results

    logger.info(f"Найдено {len(pdf_files)} PDF файлов")

    for pdf_path in pdf_files:
        logger.info(f"Обработка: {pdf_path}")
        text = extract_pdf_text(str(pdf_path))
        if not text:
            logger.warning(f"Пропущен (пустой текст): {pdf_path}")
            continue

        cleaned_text = clean_text(text)
        metadata = get_pdf_metadata(str(pdf_path))

        results.append({
            "text": cleaned_text,
            "metadata": metadata
        })
        logger.info(f"Добавлен: {pdf_path} (длина текста: {len(cleaned_text)} символов)")

    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    logger.info(f"Сохранено {len(results)} документов в {output_json}")
    return results

if __name__ == "__main__":
    process_pdf_directory(
        input_dir="./documents",
        output_json="./extracted_data.json"
    )
```

---

### 4.8. Схема ETL-процесса (Mermaid)

```mermaid
flowchart TD
    A[Исходные файлы] --> B{Тип файла}
    B -->|PDF| C1[pypdf / pdfplumber]
    B -->|DOCX| C2[python-docx]
    B -->|HTML| C3[beautifulsoup4]
    B -->|Markdown| C4[markdown]
    B -->|Excel/CSV| C5[pandas]
    B -->|Изображения| C6[OCR / Tesseract]

    C1 --> D[Извлечение текста]
    C2 --> D
    C3 --> D
    C4 --> D
    C5 --> D
    C6 --> D

    D --> E[Очистка текста]
    E --> F[Извлечение метаданных]
    F --> G[Сохранение в JSON]
    G --> H[Далее: чанкинг и векторизация]
```

---

### 4.9. Контрольные вопросы и задания

**Вопросы для самопроверки:**

1. *Почему pdfplumber лучше подходит для извлечения таблиц из PDF, чем pypdf?*  
   **Ответ:** pdfplumber анализирует позиционирование букв и графические элементы, что позволяет определять границы ячеек и восстанавливать структуру таблицы. pypdf извлекает текст по потокам без учёта позиционирования.

2. *В каких случаях следует использовать стемминг вместо лемматизации?*  
   **Ответ:** Стемминг быстрее и проще, поэтому он подходит для лексического поиска (BM25) в высоконагруженных системах, где скорость критична. Лемматизация точнее, поэтому используется для задач, где важна семантическая точность, но она медленнее.

3. *Почему важно сохранять метаданные документов в RAG-системе?*  
   **Ответ:** Метаданные позволяют фильтровать документы при поиске (например, по дате, автору, категории), цитировать источники в ответе и управлять версиями (помечать устаревшие документы).

**Практическое задание:**

Модифицируйте скрипт из раздела 4.7 так, чтобы он обрабатывал **.docx и .html** файлы. Добавьте логирование количества успешно обработанных документов каждого типа. Результат должен сохраняться в тот же JSON-формат с полем `"type"`, указывающим исходный формат.

---

### 4.10. Список литературы

1. **pypdf Documentation.** – https://pypdf.readthedocs.io/ – официальная документация по работе с PDF.
2. **pdfplumber Documentation.** – https://pdfplumber.readthedocs.io/ – работа с PDF и таблицами.
3. **python-docx Documentation.** – https://python-docx.readthedocs.io/ – извлечение текста из .docx.
4. **BeautifulSoup Documentation.** – https://www.crummy.com/software/BeautifulSoup/ – парсинг HTML.
5. **pymorphy2 Documentation.** – https://pymorphy2.readthedocs.io/ – морфологический анализ русского языка.
6. **spaCy Documentation.** – https://spacy.io/ – лемматизация для английского и других языков.
7. **Airflow Documentation.** – https://airflow.apache.org/ – управление ETL-пайплайнами.



## Тема 5. Чанкинг (разбиение текста)

После того как мы извлекли и очистили текст из документов, следующий критический этап — **разбиение на чанки (chunking)**. Это один из самых важных и недооценённых этапов в построении RAG-системы. Качество поиска напрямую зависит от того, как мы разрежем документы на фрагменты: слишком маленький чанк теряет контекст, слишком большой — вносит шум и размывает семантику. В этой теме мы разберём все стратегии чанкинга, их математические основы, параметры и практическую реализацию, а также проведём эксперимент для выбора оптимальных настроек.

---

### 5.1. Что такое чанкинг и зачем он нужен

**Чанк (chunk)** — это фрагмент текста (предложение, абзац, смысловой блок), который подаётся на вход ретриверу для генерации эмбеддинга и последующего поиска. Весь документ разбивается на множество чанков, каждый из которых становится отдельной единицей в векторной базе данных.

#### 5.1.1. Почему нельзя использовать весь документ целиком?

Большинство LLM имеют ограничение на длину контекстного окна (например, 4096, 8192 или 128K токенов), но даже если окно достаточно большое, есть три причины для чанкинга:

1. **Точность поиска.** Если чанк слишком большой, он содержит много разнородной информации. Векторное представление такого чанка будет усреднённым, и поиск по конкретному запросу станет менее точным.
2. **Качество эмбеддингов.** Модели эмбеддингов имеют ограничение на длину входного текста (обычно 512 токенов для BERT-подобных моделей, до 1024 для современных). Обрезание или усреднение длинных текстов снижает качество.
3. **Шум.** Большой чанк содержит много нерелевантной информации, которая может «забить» сигнал и привести к ложным срабатываниям.

#### 5.1.2. Оптимальный размер чанка

Эмпирическое правило: **200–1000 токенов** на чанк.

| Размер чанка (токенов) | Преимущества | Недостатки |
|------------------------|--------------|------------|
| **< 200** | Высокая точность, много чанков | Теряется контекст, много шума |
| **200–500** | Хороший баланс | Требует настройки overlap |
| **500–1000** | Больше контекста | Может быть шумно, медленнее поиск |
| **> 1000** | Минимум чанков | Низкая точность, проблемы с эмбеддингами |

**Математическое обоснование:** Количество чанков для документа из $N$ токенов при размере чанка $S$ и перекрытии $O$:

$$
\text{NumChunks} = \left\lceil \frac{N - O}{S - O} \right\rceil
$$

Например, для документа 10 000 токенов, $S = 500$, $O = 50$:
- Количество чанков = $\lceil (10000 - 50) / (500 - 50) \rceil = \lceil 9950 / 450 \rceil = \lceil 22.11 \rceil = 23$ чанка.

#### 5.1.3. Связь с контекстным окном LLM

Суммарная длина всех чанков, передаваемых в LLM, не должна превышать контекстное окно. Если модель имеет окно 4096 токенов, а мы передаём 5 чанков по 500 токенов с промптом и ответом, то общая длина ≈ 5 × 500 + 500 (промпт) + 500 (ответ) = 3500 токенов, что влезает в окно. Если нужно больше чанков — используем модели с большим контекстным окном (например, 128K).

---

### 5.2. Стратегии разбиения текста

Существует несколько подходов к чанкингу, каждый со своими сильными и слабыми сторонами. Выбор стратегии зависит от структуры документов и требований к качеству.

#### 5.2.1. Fixed-size chunking (Фиксированный размер)

**Принцип:** текст разбивается на фрагменты строго фиксированного размера (например, по 500 символов или токенов) без учёта структуры.

**Схема:**
```
[Предложение 1] [Предложение 2] [Предложение 3] [Предложение 4] [Предложение 5]
      ↓                    ↓                    ↓                    ↓
  Чанк 1 (500 симв)    Чанк 2 (500 симв)    Чанк 3 (500 симв)    Чанк 4 (500 симв)
```

**Преимущества:** простой, быстрый, предсказуемый.
**Недостатки:** разрывает предложения, абзацы и смысловые блоки, теряет структуру документа.

**Когда использовать:** для однородных текстов без сложной структуры (например, логи файлов, сырые данные).

#### 5.2.2. Recursive chunking (Рекурсивное разбиение)

**Принцип:** текст разбивается иерархически по разделителям: сначала по самым крупным (например, абзацы), затем, если чанк всё ещё слишком большой, по предложениям, затем по словам. Это самый популярный и эффективный метод.

**Схема:**
```
[Документ]
    ↓
[Абзац 1] [Абзац 2] [Абзац 3]  ← разбиение по \n\n
    ↓         ↓
[Предложение 1] [Предложение 2] ← разбиение по \n, . , !
    ↓
[Слово 1] [Слово 2]            ← разбиение по пробелам (если нужно)
```

**Пример иерархии разделителей:**
1. `\n\n` (двойной перевод строки — абзац)
2. `\n` (один перевод строки)
3. `. ` (точка с пробелом — предложение)
4. `! ` (восклицательный знак)
5. `? ` (вопросительный знак)
6. `, ` (запятая)
7. пробел

**Преимущества:** сохраняет структуру документа, не разрывает предложения без необходимости, адаптивен к разным типам текстов.
**Недостатки:** сложнее в реализации, требует тщательного выбора разделителей.

**Когда использовать:** для любых структурированных текстов — статьи, документация, книги, отчёты.

#### 5.2.3. Semantic chunking (Смысловое разбиение)

**Принцип:** чанки определяются не по длине, а по смысловым границам — смене темы, завершённости мысли. Для этого используется анализ эмбеддингов: текст разбивается на предложения, затем каждое предложение кодируется, и границы проводятся там, где косинусное расстояние между соседними предложениями превышает порог.

**Схема:**
```
[Предложение 1] [Предложение 2] [Предложение 3] [Предложение 4] [Предложение 5]
     ↓                ↓                ↓                ↓                ↓
   Эмбеддинг 1      Эмбеддинг 2      Эмбеддинг 3      Эмбеддинг 4      Эмбеддинг 5
     ↓                ↓                ↓                ↓                ↓
   sim(1,2)=0.95   sim(2,3)=0.92    sim(3,4)=0.45    sim(4,5)=0.90
                                           ↑
                                    Смена темы! Граница чанка.
```

**Преимущества:** чанки семантически цельны, что улучшает качество поиска.
**Недостатки:** ресурсоёмко (нужно вычислять эмбеддинги для каждого предложения), сложно подобрать порог.

**Когда использовать:** для сложных, многотемных документов, где важна смысловая целостность (научные статьи, книги).

#### 5.2.4. Sliding window с перекрытием (overlap)

**Принцип:** каждый следующий чанк начинается не с конца предыдущего, а немного раньше, захватывая часть предыдущего чанка. Это предотвращает потерю информации на границах.

**Схема:**
```
[Чанк 1]                   [Чанк 3]
  [Чанк 2]                   [Чанк 4]
    ↓                         ↓
  overlap (10–20%)          overlap (10–20%)
```

**Преимущества:** сохраняет контекст на границах, уменьшает риск потери важной информации.
**Недостатки:** увеличивает количество чанков и избыточность.

**Когда использовать:** всегда в сочетании с другими стратегиями, особенно для текстов с длинными предложениями.

#### 5.2.5. Sentence‑based / Paragraph‑based (По предложениям/абзацам)

**Принцип:** разбиение по естественным границам — предложениям или абзацам.

**Преимущества:** максимально сохраняет структуру, идеально для диалогов и юридических документов.
**Недостатки:** размер чанков непредсказуем (может быть слишком большим или слишком маленьким).

**Когда использовать:** для документов с чёткой структурой (законы, инструкции, диалоги).

---

### 5.3. Параметры чанкинга

#### 5.3.1. Выбор `chunk_size`

- Для BERT-подобных моделей (384/768d): **256–512 токенов**.
- Для современных моделей (multilingual-e5, BGE): **512–1024 токенов**.
- Если модель поддерживает длинные последовательности (например, 8192 токенов), можно увеличить до 1024–2048 токенов.

**Практический совет:** начните с 500 токенов и экспериментируйте.

#### 5.3.2. Выбор `chunk_overlap`

- **Стандарт:** 10–20% от размера чанка.
- Для S=500: overlap = 50–100 токенов.
- Для S=1000: overlap = 100–200 токенов.

**Зачем он нужен:** предложение на границе двух чанков может быть разорвано. Overlap гарантирует, что оно попадёт в оба чанка целиком, и при поиске не будет потеряно.

#### 5.3.3. Обработка коротких и длинных чанков

- **Короткие чанки (< 50 токенов):** объединять с соседними (если они не являются заголовками или отдельными пунктами).
- **Длинные чанки (> max_size):** разбивать рекурсивно, используя более мелкие разделители.

#### 5.3.4. Сохранение структуры документа

Важно сохранять заголовки, маркированные списки и таблицы при разбиении. Например, можно добавить заголовок раздела в метаданные каждого чанка, чтобы сохранить контекст.

---

### 5.4. Практическая реализация

#### 5.4.1. Реализация рекурсивного сплиттера на Python

```python
import re
from typing import List, Tuple

class RecursiveTextSplitter:
    """
    Рекурсивный сплиттер с иерархией разделителей.
    """
    def __init__(self, chunk_size: int = 500, chunk_overlap: int = 50):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.separators = ["\n\n", "\n", ". ", "! ", "? ", ", ", " "]

    def split_text(self, text: str) -> List[str]:
        """Разбивает текст на чанки."""
        chunks = []
        current_chunk = []
        current_len = 0

        # Рекурсивное разбиение по разделителям
        segments = self._split_by_separators(text, self.separators)

        for segment in segments:
            seg_len = len(segment)
            # Если текущий чанк + новый сегмент > max_size
            if current_len + seg_len > self.chunk_size and current_chunk:
                chunks.append("".join(current_chunk).strip())
                # Overlap: сохраняем часть предыдущего чанка
                overlap_text = self._get_overlap("".join(current_chunk), self.chunk_overlap)
                current_chunk = [overlap_text]
                current_len = len(overlap_text)

            current_chunk.append(segment)
            current_len += seg_len

        if current_chunk:
            chunks.append("".join(current_chunk).strip())

        return chunks

    def _split_by_separators(self, text: str, separators: List[str]) -> List[str]:
        """Рекурсивно разбивает текст по разделителям."""
        if not text:
            return []

        separator = separators[0]
        remaining_seps = separators[1:]

        if not remaining_seps:
            # Последний разделитель — пробел
            return text.split(separator)

        parts = text.split(separator)
        result = []
        for i, part in enumerate(parts):
            if len(part) <= self.chunk_size:
                result.append(part)
            else:
                # Рекурсивно разбиваем более мелким разделителем
                sub_parts = self._split_by_separators(part, remaining_seps)
                result.extend(sub_parts)

            if i < len(parts) - 1:
                result.append(separator)  # Добавляем разделитель обратно

        return result

    def _get_overlap(self, text: str, overlap_len: int) -> str:
        """Возвращает последние `overlap_len` символов текста."""
        return text[-overlap_len:] if len(text) > overlap_len else text

# Пример использования
splitter = RecursiveTextSplitter(chunk_size=500, chunk_overlap=50)
text = """Это первый абзац. Он содержит несколько предложений. Здесь есть информация о налогах.

Это второй абзац. Он начинается с новой темы. Здесь говорится о сборах.
"""
chunks = splitter.split_text(text)
for i, chunk in enumerate(chunks):
    print(f"Чанк {i+1}: {chunk[:100]}...")
```

#### 5.4.2. Использование LangChain

```python
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", "! ", "? ", ", ", " "],
    length_function=len,  # или можно использовать count_tokens
)
chunks = splitter.split_text(text)
```

#### 5.4.3. Использование NLTK и spaCy для сплиттинга по предложениям

```python
import nltk
nltk.download('punkt')
from nltk.tokenize import sent_tokenize

def chunk_by_sentences(text: str, max_sentences: int = 3) -> List[str]:
    """Разбивает текст на чанки по предложениям."""
    sentences = sent_tokenize(text)
    chunks = []
    for i in range(0, len(sentences), max_sentences):
        chunk = " ".join(sentences[i:i+max_sentences])
        chunks.append(chunk)
    return chunks
```

#### 5.4.4. Сравнение стратегий на примере

Возьмём текст из 5000 слов (примерно 7500 токенов). Результаты:

| Стратегия | Размер | Overlap | Количество чанков | Пример чанка |
|-----------|--------|---------|-------------------|--------------|
| Fixed-size | 500 токенов | 0 | 15 | "...разрывает предложения..." |
| Recursive | 500 токенов | 50 | 16 | "...сохраняет структуру..." |
| Semantic | переменный | 0 | 12 | "...смысловая целостность..." |
| Sentence-based | ~5 предложений | 0 | 14 | "...целые предложения..." |

---

### 5.5. Эксперимент: сравнение размеров чанков

**Цель:** определить оптимальный размер чанка для вашей задачи.

**Методика:**
1. Возьмите один документ (например, статью из 10 000 токенов).
2. Разбейте его тремя способами:
   - **Маленькие чанки:** S=200, O=20
   - **Средние чанки:** S=500, O=50
   - **Большие чанки:** S=1000, O=100
3. Для каждого способа подсчитайте количество чанков и приведите пример одного чанка.

**Пример результатов:**

| Размер чанка (S) | Overlap (O) | Количество чанков | Пример чанка (первые 100 символов) |
|------------------|-------------|-------------------|------------------------------------|
| **200** | 20 | 52 | "Налог на прибыль организаций... | Ставка 20%..." |
| **500** | 50 | 21 | "Налог на прибыль организаций регулируется главой 25 НК РФ. Ставка 20%... " |
| **1000** | 100 | 11 | "Налог на прибыль организаций регулируется главой 25 НК РФ. Ставка 20%... Особенности расчёта для ИТ-компаний..." |

**Вывод:** для данного текста размер 500 токенов даёт хороший баланс: чанки достаточно короткие для точного поиска, но содержат достаточно контекста для генерации ответа.

---

### 5.6. Контрольные вопросы

1. *Почему слишком маленький чанк (менее 100 токенов) ухудшает качество поиска?*  
   **Ответ:** Маленький чанк содержит недостаточно контекста. Эмбеддинг такого чанка будет неполным, и запрос, требующий более широкого контекста, не сможет найти нужный фрагмент. Кроме того, увеличивается количество чанков, что замедляет поиск.

2. *В чём преимущество рекурсивного сплиттера перед фиксированным?*  
   **Ответ:** Рекурсивный сплиттер учитывает структуру документа и не разрывает предложения и абзацы без необходимости. Это сохраняет смысловую целостность чанков и улучшает качество эмбеддингов и поиска.

3. *Какой overlap рекомендуется использовать и почему?*  
   **Ответ:** Рекомендуется overlap = 10–20% от размера чанка. Это гарантирует, что предложения на границе двух чанков не будут потеряны, так как они попадут в оба чанка целиком.

---

### 5.7. Задания

1. **Реализация рекурсивного сплиттера.** Возьмите любой текст из 3–5 абзацев и примените к нему рекурсивный сплиттер с параметрами `chunk_size=300`, `chunk_overlap=30`. Выведите все полученные чанки и прокомментируйте качество разбиения.

2. **Эксперимент с размерами.** Сравните качество поиска при разных размерах чанков. Возьмите корпус из 5 документов и 5 запросов. Для каждого размера чанка (200, 500, 1000 токенов) выполните поиск и оцените релевантность найденных документов. Сделайте отчёт с выводами.

---

### 5.8. Список литературы

1. **LangChain Documentation.** *Text Splitters*. – https://python.langchain.com/docs/modules/data_connection/document_transformers/
2. **NLTK Documentation.** *Tokenization*. – https://www.nltk.org/api/nltk.tokenize.html
3. **spaCy Documentation.** *Sentence Segmentation*. – https://spacy.io/api/sentencizer
4. **Gao, Y., et al. (2023).** *Retrieval-Augmented Generation for Large Language Models: A Survey*. – раздел про обработку данных.
